In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, re, glob, shutil, warnings
import numpy as np, pandas as pd, xarray as xr
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ACC_PARQUET = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
GMTDS_ORG   = rf'{DATA_ROOT}/climate_AIS_data/GMTDS/cargo_ship/organized'
WAVE_DIR    = rf'{DATA_ROOT}/AIS_processing/weather_data/wave'
WIND_DIR    = rf'{DATA_ROOT}/AIS_processing/weather_data/wind'
CACHE_BASE  = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2'

GRID_SIZE = 2.0                       # main 2.0; rerun with 1.0 for sensitivity
YEARS     = list(range(2011, 2022))
DECIMATE  = 4
TRAFFIC_FLOOR = 0.5

# ── final 6 region bounds (S, N, W, E), multiple sub-boxes, non-overlapping ← confirm this is the intended version ──
REGIONS = {
    'NW_Europe': {
        'Iberia_Atlantic':   [(36, 45, -15, -6)],
        'Biscay_Celtic':     [(45, 63, -15, -6), (43, 63, -6, 0)],
        'North_Sea':         [(48, 63, 0, 13)],
    },
    'NW_Atlantic': {
        'GoM_Caribbean': [(18, 31, -98, -60)],
        'NE_Coast':      [(31, 42, -82, -60), (42, 50, -67, -60)],
        'Newfoundland':  [(45, 55, -60, -46)],
    },
    'Asia': {
        'SE_Asia': [(-10, 20, 95, 130)],
        'E_Asia':  [(20, 46, 117, 146)],
    },
    'Med_Black': {
        'Med_West': [(30, 43, -6, 0), (30, 46, 0, 19)],
        'Med_East': [(30, 46, 19, 26.5), (30, 41, 26.5, 37)],
        'Black':    [(41, 47.5, 26.5, 42)],
    },
    'Baltic': {
        'Baltic': [(53, 66, 13, 30)],
    },
    'Indian_Ocean_N': {
        'Arabian': [(5, 30, 50, 78)],
        'BoB':     [(5, 26, 78, 95)],
    },
}
TYPE_FILTERS = {
    'Collision':      r'collision|contact',
    'Stranding':      r'strand|ground|shoal|beach',
    'Machinery':      r'machin|engine|propul',
    'Foundering':     r'founder|submer',
    'Fire_Explosion': r'fire|explo|burn',
    'Hull_Damage':    r'hull|holed|crack|structur',
}

CACHE = os.path.join(CACHE_BASE, f'grid{GRID_SIZE:g}')
os.makedirs(CACHE, exist_ok=True)
def cpath(region, name): return os.path.join(CACHE, f'{region}__{name}.parquet')

def to_cell(lat, lon, G=GRID_SIZE):
    return np.round(np.floor(lat/G)*G+G/2, 3), np.round(np.floor(lon/G)*G+G/2, 3)
def gid(clat, clon): return f'{clat:.3f}_{clon:.3f}'
def assign_region(lat, lon):
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            if (S <= lat < N) and (W <= lon < E): return r
    return None
def region_bounds(region):
    bs = REGIONS[region]
    return (min(b[0] for b in bs), max(b[1] for b in bs),
            min(b[2] for b in bs), max(b[3] for b in bs))
def draw_region_boxes(ax, color='k', **kw):
    st = dict(fill=False, edgecolor=color, lw=.8, ls='--'); st.update(kw)
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            ax.add_patch(plt.Rectangle((W, S), E-W, N-S, **st))

print('CELL1 OK |', CACHE)


In [ ]:
# ============================================================
# M2 panel · CELL 1 — config / region bounds / helpers / accident type and consequence definitions
# ============================================================
import os, re, glob, shutil, itertools, warnings
import numpy as np, pandas as pd, xarray as xr
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ACC_PARQUET = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
GMTDS_ORG   = rf'{DATA_ROOT}/climate_AIS_data/GMTDS/cargo_ship/organized'
WAVE_DIR    = rf'{DATA_ROOT}/AIS_processing/weather_data/wave'   # era5_swh_{y}.nc
WIND_DIR    = rf'{DATA_ROOT}/AIS_processing/weather_data/wind'   # era5_wind10m_{y}.grib (actually NetCDF)
CACHE_BASE  = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2'

GRID_SIZE     = 2.0                     # main 2.0; rerun with 1.0 for sensitivity
YEARS         = list(range(2011, 2022))
DECIMATE      = 4
TRAFFIC_FLOOR = 0.5

# ── 6 regions / 15 sub-regions (S, N, W, E), globally non-overlapping ──
REGIONS = {
    'NW_Europe': {
        'Iberia_Atlantic':   [(36, 45, -15, -6)],
        'Biscay_Celtic':     [(45, 63, -15, -6), (43, 63, -6, 0)],
        'North_Sea':         [(48, 63, 0, 13)],
    },
    'NW_Atlantic': {
        'GoM_Caribbean': [(18, 31, -98, -60)],
        'NE_Coast':      [(31, 42, -82, -60), (42, 50, -67, -60)],
        'Newfoundland':  [(45, 55, -60, -46)],
    },
    'Asia': {
        'SE_Asia': [(-10, 20, 95, 130)],
        'E_Asia':  [(20, 46, 117, 146)],
    },
    'Med_Black': {
        'Med_West': [(30, 43, -6, 0), (30, 46, 0, 19)],
        'Med_East': [(30, 46, 19, 26.5), (30, 41, 26.5, 37)],
        'Black':    [(41, 47.5, 26.5, 42)],
    },
    'Baltic': {
        'Baltic': [(53, 66, 13, 30)],
    },
    'Indian_Ocean_N': {
        'Arabian': [(5, 30, 50, 78)],
        'BoB':     [(5, 26, 78, 95)],
    },
}
FLAT_BOXES = [(r, s, b) for r, subs in REGIONS.items() for s, bs in subs.items() for b in bs]
SUBREGIONS = [s for r, subs in REGIONS.items() for s in subs]

CACHE = os.path.join(CACHE_BASE, f'grid{GRID_SIZE:g}')
os.makedirs(CACHE, exist_ok=True)
def cpath(region, name): return os.path.join(CACHE, f'{region}__{name}.parquet')

def to_cell(lat, lon, G=GRID_SIZE):
    return np.round(np.floor(lat/G)*G + G/2, 3), np.round(np.floor(lon/G)*G + G/2, 3)
def gid(clat, clon): return f'{clat:.3f}_{clon:.3f}'

def assign_rs(lat, lon):
    for r, s, (S, N, W, E) in FLAT_BOXES:
        if S <= lat < N and W <= lon < E:
            return r, s
    return None, None
def assign_region(lat, lon):    return assign_rs(lat, lon)[0]
def assign_subregion(lat, lon): return assign_rs(lat, lon)[1]

def boxes_of(region, sub=None):
    if sub is not None: return list(REGIONS[region][sub])
    return [b for s in REGIONS[region] for b in REGIONS[region][s]]
def region_bounds(region, sub=None):
    bs = boxes_of(region, sub)
    return (min(b[0] for b in bs), max(b[1] for b in bs),
            min(b[2] for b in bs), max(b[3] for b in bs))
def draw_region_boxes(ax, color='k', **kw):
    st = dict(fill=False, edgecolor=color, lw=.8, ls='--'); st.update(kw)
    for r, s, (S, N, W, E) in FLAT_BOXES:
        ax.add_patch(plt.Rectangle((W, S), E-W, N-S, **st))

_ov = [(f'{r1}/{s1}{b1}', f'{r2}/{s2}{b2}')
       for (r1,s1,b1), (r2,s2,b2) in itertools.combinations(FLAT_BOXES, 2)
       if min(b1[1],b2[1]) > max(b1[0],b2[0]) and min(b1[3],b2[3]) > max(b1[2],b2[2])]
assert not _ov, f'region bounds overlap: {_ov}'

# ============================================================
# accident type: cause_subcategory = MECE primary class; 14 atomic cause tokens = any mention
# ============================================================
PRIMARY_MAP = {
    'Machinery damage': 'Machinery',
    'grounding':        'Grounding',
    'collision':        'Collision',
    'contact':          'Contact',          # split from Collision
    'Fire/explosion':   'Fire_Explosion',
    'Foundered':        'Foundered',
    'Hull damage':      'Hull_Damage',
}
NONWX_RGX     = r'piracy|arrests|seizure|labour dispute|detained|port state|missing/overdue|war loss'
TYPES_WX      = ['Grounding', 'Collision', 'Contact', 'Foundered', 'Hull_Damage']
TYPES_PLACEBO = ['Machinery', 'Piracy_Other']
TYPES_ALL     = list(dict.fromkeys(list(PRIMARY_MAP.values()) + ['Piracy_Other', 'Misc']))

CAUSE_TOKEN = {
    'Wrecked/stranded (aground)':                                    'ANY_grounding',
    'Collision (involving vessels)':                                 'ANY_collision',
    'Contact (eg. Harbour wall)':                                    'ANY_contact',
    'Hull damage (holed, cracks, structural failure)':               'ANY_hull',
    'Foundered (sunk, submerged)':                                   'ANY_founder',
    'Fire/explosion':                                                'ANY_fire',
    'Machinery damage/failure (e.g. lost rudder, fouled propellor)': 'ANY_machinery',
    'Piracy':                                                        'ANY_piracy',
}
NONWX_TOKENS = {'Miscellaneous', 'Labour dispute', 'Arrests/seizures',
                'Vessel detained for Ports state control regulation',
                'Missing/overdue', 'War loss/damage during hostilities'}

def split_causes(s):
    """'A (x, y), B' -> ['A (x, y)', 'B']; commas inside parentheses are not separators"""
    out, buf, depth = [], '', 0
    for ch in str(s):
        if ch == '(': depth += 1
        elif ch == ')': depth -= 1
        if ch == ',' and depth == 0:
            if buf.strip(): out.append(buf.strip())
            buf = ''
        else:
            buf += ch
    if buf.strip(): out.append(buf.strip())
    return out

def label_acc_type(df):
    t = df['cause_subcategory'].fillna('other').map(PRIMARY_MAP)
    is_nw = df['cause'].fillna('').str.lower().str.contains(NONWX_RGX, regex=True)
    return t.fillna(pd.Series(np.where(is_nw, 'Piracy_Other', 'Misc'), index=df.index))

def add_any_flags(df):
    sets = df['cause'].fillna('').map(lambda s: set(split_causes(s)))
    for tok, col in CAUSE_TOKEN.items():
        df[col] = sets.map(lambda S, t=tok: int(t in S))
    df['n_causes'] = sets.map(len).astype(int)
    unk = (set().union(*sets) if len(sets) else set()) - set(CAUSE_TOKEN) - NONWX_TOKENS
    if unk: print('⚠ unregistered cause tokens:', unk)
    return df

CNT_COLS = (['A_it', 'A_wx', 'A_placebo']
            + [f'A_{t}' for t in TYPES_ALL]
            + list(CAUSE_TOKEN.values()) + ['n_causes']
            + ['A_serious', 'A_wx_serious', 'A_tloss', 'A_tloss_official', 'A_actual_loss', 'A_poll']
            + ['n_dead_sum', 'n_injured_sum', 'n_missing_sum', 'n_lifelost_sum',
               'A_fatal', 'A_injury', 'A_cas_reported', 'A_cas_zero']
            + ['dwt_sum', 'gt_sum', 'A_dwt_known'])

print(f'CELL1 OK | GRID={GRID_SIZE} | {len(REGIONS)} regions / {len(SUBREGIONS)} sub-regions | cache={CACHE}')


In [ ]:
shutil.rmtree(CACHE, ignore_errors=True); os.makedirs(CACHE, exist_ok=True)
print('cache cleared:', CACHE)


In [ ]:
mani = pd.read_csv(os.path.join(GMTDS_ORG, '_manifest.csv'))
mani.columns = [c.strip().lstrip('\ufeff') for c in mani.columns]

pat  = re.compile(r'Cargo_b(\d+)_(\d{4})_(\d{1,2})\.tif$', re.I)
rows = []
for p in glob.glob(os.path.join(GMTDS_ORG, '**', '*.tif'), recursive=True):
    m = pat.search(os.path.basename(p))
    if m and int(m.group(2)) in YEARS:
        rows.append((int(m.group(1)), int(m.group(2)), int(m.group(3)), p))
tif_idx = (pd.DataFrame(rows, columns=['box_id','year','month','path'])
             .drop_duplicates(['box_id','year','month']))

# ⚠ do not use the manifest regions labels (old region names; Asia / Indian_Ocean_N return nothing)
def boxes_of_region(region):
    out = []
    for t in mani.itertuples():
        if any(min(t.N, N) > max(t.S, S) and min(t.E, E) > max(t.W, W)
               for (S, N, W, E) in boxes_of(region)):
            out.append(int(t.box_id))
    return sorted(set(out))

print('CELL2 OK | tif entries', len(tif_idx))
for _r in REGIONS:
    _b = boxes_of_region(_r)
    _n = tif_idx[tif_idx.box_id.isin(_b)].groupby(['year','month']).ngroups
    print(f'  {_r:16s} boxes={_b}  months covered={_n}/{len(YEARS)*12}')
    assert _b, f'{_r} matched no GMTDS box!'


In [ ]:
def build_sea_cells():
    cp = os.path.join(CACHE, '_sea_cells.parquet')
    if os.path.exists(cp): return pd.read_parquet(cp)

    ds = xr.open_dataset(os.path.join(WAVE_DIR, f'era5_swh_{YEARS[0]}.nc'))
    la, lo = ds['latitude'].values, ds['longitude'].values
    lo180  = np.where(lo > 180, lo - 360, lo)
    ocean  = ~np.isnan(ds['swh'].isel(valid_time=0).values); ds.close()

    LO, LA   = np.meshgrid(lo180, la)
    CLA, CLO = to_cell(LA, LO)
    cells = pd.DataFrame({'grid_lat': CLA[ocean], 'grid_lon': CLO[ocean]}).drop_duplicates()

    # ★ assign region by [grid cell centre], not by ERA5 points, so grid_id -> (region, subregion)
    #   is a function; accident and grid sides always give the same label. Cells centred outside all boxes are dropped.
    rs = [assign_rs(a, o) for a, o in zip(cells.grid_lat, cells.grid_lon)]
    cells['region']    = [x[0] for x in rs]
    cells['subregion'] = [x[1] for x in rs]
    cells = cells[cells.region.notna()].copy()
    cells['grid_id'] = (cells.grid_lat.map('{:.3f}'.format) + '_' +
                        cells.grid_lon.map('{:.3f}'.format))

    # ★ intersect with existing dmax to avoid re-reading ERA5 for a few cells (only 1 cell differs)
    have = set()
    for r in REGIONS:
        p = cpath(r, 'dmax_wave')
        if os.path.exists(p):
            have |= set(pd.read_parquet(p, columns=['grid_id']).grid_id.unique())
    if have:
        n0 = len(cells)
        drop = cells[~cells.grid_id.isin(have)]
        cells = cells[cells.grid_id.isin(have)].copy()
        if len(drop):
            print(f'intersect with existing dmax: {n0} → {len(cells)} cells, dropped:')
            print(drop[['region','subregion','grid_lat','grid_lon']].to_string(index=False))

    cells = cells[['region','subregion','grid_id','grid_lat','grid_lon']].reset_index(drop=True)
    assert cells.grid_id.is_unique, 'grid_id not unique'
    assert cells.subregion.nunique() == len(SUBREGIONS), \
           f'missing sub-regions: {set(SUBREGIONS) - set(cells.subregion)}'
    cells.to_parquet(cp, index=False)
    return cells

sea_cells = build_sea_cells()
CELL2RS = sea_cells.set_index('grid_id')[['region','subregion']]   # globally unique cell→region mapping
SUBMAP  = dict(zip(sea_cells.grid_id, sea_cells.subregion))
print('sea cells total', len(sea_cells), '| sub-regions', sea_cells.subregion.nunique())
print(sea_cells.groupby(['region','subregion']).size().to_string())

fig, ax = plt.subplots(figsize=(14, 6))
for s, g in sea_cells.groupby('subregion'):
    ax.scatter(g.grid_lon, g.grid_lat, s=8, label=s)
draw_region_boxes(ax); ax.legend(loc='lower left', ncol=4, fontsize=7); ax.grid(alpha=.3)
ax.set_title(f'sea cells (grid={GRID_SIZE}) n={len(sea_cells)}')
plt.tight_layout(); plt.show()


In [ ]:
def build_Ait():
    cp = os.path.join(CACHE, '_Ait.parquet')
    if os.path.exists(cp): return pd.read_parquet(cp)

    df = pd.read_parquet(ACC_PARQUET).dropna(subset=['lat', 'lon'])
    df = df[df.year.between(YEARS[0], YEARS[-1])].copy()

    # ★ assign to cell first, then look up region via CELL2RS — same table as sea_cells, labels cannot conflict
    clat, clon = to_cell(df.lat.values, df.lon.values)
    df['grid_id'] = [gid(a, o) for a, o in zip(clat, clon)]
    n0 = len(df)
    df = df.join(CELL2RS, on='grid_id')
    df = df[df.region.notna()].copy()
    print(f'  accidents {n0:,} → in study-area sea cells {len(df):,} (dropped {n0-len(df):,}: outside regions or land cells)')

    df['year_month'] = pd.to_datetime(df['casualty_date']).dt.strftime('%Y-%m')

    # ---- type ----
    df['acc_type'] = label_acc_type(df)
    df = add_any_flags(df)
    df['A_it']      = 1
    df['A_wx']      = df.acc_type.isin(TYPES_WX).astype(int)
    df['A_placebo'] = df.acc_type.isin(TYPES_PLACEBO).astype(int)
    for t in TYPES_ALL:
        df[f'A_{t}'] = (df.acc_type == t).astype(int)

    # ---- consequence: severity / pollution / total loss ----
    df['A_serious']        = df.serious.astype(int)
    df['A_wx_serious']     = (df.A_wx.astype(bool) & df.serious).astype(int)
    df['A_poll']           = df.pollution.astype(int)
    df['A_tloss_official'] = df.loss_type.notna().astype(int)               # strict definition (sparse)
    df['A_actual_loss']    = (df.loss_type == 'Actual total loss').astype(int)
    df['A_tloss']          = (df.loss_type.notna() |
                              (df.acc_type == 'Foundered')).astype(int)     # proxy definition (recommended)

    # ---- consequence: casualties (raw fields 98.7-99.3% missing, always use with A_cas_reported) ----
    dead, inj, miss = df.n_dead, df.n_injured, df.n_missing
    df['n_dead_sum']     = dead.fillna(0)
    df['n_injured_sum']  = inj.fillna(0)
    df['n_missing_sum']  = miss.fillna(0)
    df['n_lifelost_sum'] = dead.fillna(0) + miss.fillna(0)
    df['A_fatal']        = ((dead.fillna(0) + miss.fillna(0)) > 0).astype(int)
    df['A_injury']       = (inj.fillna(0) > 0).astype(int)
    _rep = dead.notna() | inj.notna() | miss.notna()
    df['A_cas_reported'] = _rep.astype(int)                                  # reporting denominator
    df['A_cas_zero']     = (_rep & (dead.fillna(0)+inj.fillna(0)+miss.fillna(0) == 0)).astype(int)

    # ---- ship size ----
    df['dwt_sum']     = df.DWT.clip(lower=0)
    df['gt_sum']      = df.GT.clip(lower=0)
    df['A_dwt_known'] = (df.DWT > 0).astype(int)

    Ait = (df.groupby(['region', 'subregion', 'grid_id', 'year_month'])[CNT_COLS]
             .sum().reset_index())
    Ait.to_parquet(cp, index=False)
    return Ait

Ait = build_Ait()
_chk = {t: int(Ait[f'A_{t}'].sum()) for t in TYPES_ALL}
assert sum(_chk.values()) == int(Ait.A_it.sum()), 'MECE classification does not close!'
print(f'A_it {len(Ait)} rows | accidents {int(Ait.A_it.sum())} | weather-related {int(Ait.A_wx.sum())} '
      f'| placebo {int(Ait.A_placebo.sum())}')
print('MECE:', _chk)
print('any mention:', {c: int(Ait[c].sum()) for c in CAUSE_TOKEN.values()})
print('consequences:', {c: int(Ait[c].sum()) for c in
                ['A_serious','A_wx_serious','A_tloss','A_tloss_official','A_actual_loss',
                 'A_poll','A_fatal','A_injury','A_cas_reported','A_cas_zero']})
print('casualty totals:', {c: float(Ait[c].sum()) for c in
                   ['n_dead_sum','n_injured_sum','n_missing_sum','n_lifelost_sum']})
print(Ait.groupby('subregion')[['A_it','A_wx','A_serious','A_fatal']]
         .sum().sort_values('A_it', ascending=False).to_string())


In [ ]:
_j = Ait[['region','subregion','grid_id']].join(CELL2RS, on='grid_id', rsuffix='_sea')
assert (_j.region == _j.region_sea).all() and (_j.subregion == _j.subregion_sea).all(), \
       'region labels of accidents and grid cells disagree'


In [ ]:
def _bin_tif_to_cells(path, G=GRID_SIZE, dec=DECIMATE):
    with rasterio.open(path) as src:
        oh, ow = max(1, src.height//dec), max(1, src.width//dec)
        arr = src.read(1, out_shape=(oh, ow), resampling=Resampling.average, masked=True)
        T   = src.transform * src.transform.scale(src.width/ow, src.height/oh)
    valid = ~arr.mask if np.ma.isMaskedArray(arr) else np.isfinite(arr)
    data  = arr.data  if np.ma.isMaskedArray(arr) else arr
    ny, nx = arr.shape
    xs = T.c + (np.arange(nx)+0.5)*T.a; ys = T.f + (np.arange(ny)+0.5)*T.e
    clon = np.floor(xs/G)*G + G/2;      clat = np.floor(ys/G)*G + G/2
    d = pd.DataFrame({'grid_lat': np.round(np.broadcast_to(clat[:,None], arr.shape)[valid], 3),
                      'grid_lon': np.round(np.broadcast_to(clon,        arr.shape)[valid], 3),
                      'v': data[valid]})
    g = d.groupby(['grid_lat','grid_lon'])['v'].agg(psum='sum', pcount='count').reset_index()
    g['grid_id'] = [gid(a, o) for a, o in zip(g.grid_lat, g.grid_lon)]
    return g

def build_Eit(region):
    cp = cpath(region, 'Eit')
    if os.path.exists(cp): return pd.read_parquet(cp)
    sea_ids = set(sea_cells[sea_cells.region == region].grid_id)
    sub = tif_idx[tif_idx.box_id.isin(boxes_of_region(region))]
    out = []
    for (y, mo), grp in sub.groupby(['year','month']):
        acc = {}
        for p in grp.path:
            try: g = _bin_tif_to_cells(p)
            except Exception as e: print('skip', os.path.basename(p), e); continue
            for r in g.itertuples(index=False):
                if r.grid_id not in sea_ids: continue
                s, c = acc.get(r.grid_id, (0.0, 0)); acc[r.grid_id] = (s + r.psum, c + r.pcount)
        for k, (s, c) in acc.items():
            out.append((region, k, f'{y}-{mo:02d}', s/c if c else np.nan))
    Eit = pd.DataFrame(out, columns=['region','grid_id','year_month','E_it'])
    Eit.to_parquet(cp, index=False); print(f'  [{region}] E_it {len(Eit)}')
    return Eit

Eit_all = pd.concat([build_Eit(r) for r in REGIONS], ignore_index=True)
nmax = len(YEARS)*12
cov  = Eit_all[Eit_all.E_it.notna()].groupby(['region','grid_id']).size().rename('n').reset_index()
cov  = sea_cells.merge(cov, on=['region','grid_id'], how='left').fillna({'n': 0})
cov['frac'] = cov.n/nmax
print(cov.groupby('subregion').frac.describe()[['mean','min','50%']].round(2).to_string())
print('cells with no traffic at all:', int((cov.n == 0).sum()))

fig, ax = plt.subplots(figsize=(14, 6))
sc = ax.scatter(cov.grid_lon, cov.grid_lat, c=cov.frac, s=14, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(sc, ax=ax, shrink=.8); draw_region_boxes(ax)
ax.set_title('GMTDS traffic monthly coverage'); plt.tight_layout(); plt.show()


In [ ]:
def _open_var(year, var):
    if var == 'wave':
        ds = xr.open_dataset(os.path.join(WAVE_DIR, f'era5_swh_{year}.nc'),
                             chunks={'valid_time': 240})
        da = ds['swh']
    else:   # .grib is actually NetCDF
        ds = xr.open_dataset(os.path.join(WIND_DIR, f'era5_wind10m_{year}.grib'),
                             engine='netcdf4', chunks={'valid_time': 240})
        da = np.sqrt(ds['u10']**2 + ds['v10']**2)
    da = da.squeeze(drop=True)
    da = da.assign_coords(longitude=(((da['longitude']+180) % 360) - 180)).sortby('longitude')
    return ds, da

def build_daily_cellmax(region, var):
    cp = cpath(region, f'dmax_{var}')
    if os.path.exists(cp): return pd.read_parquet(cp)
    S, N, W, E = region_bounds(region)
    sea_ids = set(sea_cells[sea_cells.region == region].grid_id)
    frames = []
    for y in YEARS:
        ds, da = _open_var(y, var)
        da = da.sel(latitude=slice(N, S), longitude=slice(W, E))
        dd = da.resample(valid_time='1D').max().load()
        clat = np.round(np.floor(dd.latitude.values /GRID_SIZE)*GRID_SIZE + GRID_SIZE/2, 3)
        clon = np.round(np.floor(dd.longitude.values/GRID_SIZE)*GRID_SIZE + GRID_SIZE/2, 3)
        dd = dd.assign_coords(clat=('latitude', clat), clon=('longitude', clon))
        cell = dd.groupby('clat').max().groupby('clon').max()
        df = cell.to_dataframe(name='dmax').reset_index()
        df['grid_id'] = df.clat.map('{:.3f}'.format) + '_' + df.clon.map('{:.3f}'.format)
        df = df[df.grid_id.isin(sea_ids)].copy()
        df['date'] = pd.to_datetime(df['valid_time']).dt.normalize()
        frames.append(df[['grid_id','date','dmax']]); ds.close()
        print(f'  {region}/{var} {y}', end=' ')
    out = pd.concat(frames, ignore_index=True)
    out.to_parquet(cp, index=False); print()
    return out

print('CELL6 definitions done')


In [ ]:
import glob, os
for p in glob.glob(os.path.join(CACHE, '*.parquet')):
    b = os.path.basename(p)
    if b.endswith('__dmax_wave.parquet') or b.endswith('__dmax_wind.parquet'):
        continue                       # ★ keep all, do not re-read ERA5
    os.remove(p); print('rm', b)

In [ ]:
PCTS = [('P95', 0.95), ('P99', 0.99)]

def build_extrdays(region):
    cp = cpath(region, 'extrdays')
    if os.path.exists(cp): return pd.read_parquet(cp)

    wave = build_daily_cellmax(region, 'wave').rename(columns={'dmax': 'wave'})
    wind = build_daily_cellmax(region, 'wind').rename(columns={'dmax': 'wind'})
    d = wave.merge(wind, on=['grid_id', 'date'], how='inner')

    # ★ fix ①: drop missing cell-days. SWH=NaN under sea ice; otherwise NaN>=threshold returns False,
    #           counted as a "calm day", and true compound days get misrecorded as wind_only.
    n0 = len(d)
    d = d[np.isfinite(d.wave) & np.isfinite(d.wind)].copy()
    if len(d) < n0:
        print(f'  [{region}] dropped {n0-len(d):,} missing cell-days (SWH=NaN under sea ice)')

    d['subregion'] = d.grid_id.map(SUBMAP)
    d = d[d.subregion.notna()].copy()
    d['cmonth']     = pd.to_datetime(d['date']).dt.month
    d['year_month'] = pd.to_datetime(d['date']).dt.strftime('%Y-%m')

    ccols, thr_rec = [], []
    for tag, q in PCTS:
        # threshold: sub-region × calendar month (quantile skips NaN, but they are already removed above)
        thr = d.groupby(['subregion', 'cmonth'])[['wave', 'wind']].quantile(q)
        thr.columns = ['thr_wave', 'thr_wind']
        idx = pd.MultiIndex.from_arrays([d.subregion, d.cmonth])
        tw  = thr['thr_wave'].reindex(idx).values
        ti  = thr['thr_wind'].reindex(idx).values
        ew, ei = d.wave.values >= tw, d.wind.values >= ti
        d[f'wind_only_{tag}']    = (ei & ~ew).astype('int8')
        d[f'wave_only_{tag}']    = (ew & ~ei).astype('int8')
        d[f'compound_{tag}']     = (ew &  ei).astype('int8')
        d[f'wind_or_wave_{tag}'] = (ew |  ei).astype('int8')
        ccols += [f'{c}_{tag}' for c in ['wind_only', 'wave_only', 'compound', 'wind_or_wave']]
        t = thr.reset_index(); t['pct'] = tag; t['region'] = region; thr_rec.append(t)

    # ★ self-check ①: sum of three exclusive classes == union
    for tag, _ in PCTS:
        assert (d[f'wind_only_{tag}'] + d[f'wave_only_{tag}'] + d[f'compound_{tag}']
                == d[f'wind_or_wave_{tag}']).all(), f'{region} {tag} mutual exclusivity broken'
    # ★ self-check ②: with the same quantile, exceedance days are equal for wind and wave → wind_only must equal wave_only.
    #           unequal = missing data counted as "non-extreme". This is the probe for pitfall ①.
    for tag, _ in PCTS:
        a, b = int(d[f'wind_only_{tag}'].sum()), int(d[f'wave_only_{tag}'].sum())
        assert a == b, f'{region} {tag}: wind_only({a}) != wave_only({b}), missing data still present'

    nd  = d.groupby(['grid_id', 'year_month']).size().rename('n_days_extr')
    agg = d.groupby(['grid_id', 'year_month'])[ccols].sum()
    agg = pd.concat([nd, agg], axis=1).reset_index()
    agg['region'] = region
    agg = agg.merge(sea_cells[['grid_id', 'subregion']], on='grid_id', how='left')

    agg.to_parquet(cp, index=False)
    pd.concat(thr_rec).to_parquet(cpath(region, 'thresholds'), index=False)
    print(f'  [{region}] extrdays {len(agg):,} rows')
    return agg

Extr_all = pd.concat([build_extrdays(r) for r in REGIONS], ignore_index=True)
Thr_all  = pd.concat([pd.read_parquet(cpath(r, 'thresholds')) for r in REGIONS], ignore_index=True)

print('\ntotal days by class:')
for tag, _ in PCTS:
    cols = [f'{c}_{tag}' for c in ['wind_only', 'wave_only', 'compound', 'wind_or_wave']]
    print(f'  {tag}:', {c: int(Extr_all[c].sum()) for c in cols})
print('\nP95 thresholds (sub-region × calendar month, first 12 rows):')
print(Thr_all[Thr_all.pct == 'P95'].head(12).round(2).to_string(index=False))

In [ ]:
BINS_VER = 'v4'          # bump when cut points or logic change, to avoid reading a stale cache

WAVE_BINS = [(0, 1.25), (1.25, 2.5), (2.5, 4), (4, 6), (6, 9), (9, np.inf)]   # m, WMO sea state
WIND_BINS = [(0, 8.0), (8.0, 10.8), (10.8, 13.9), (13.9, 17.2),
             (17.2, 20.8), (20.8, np.inf)]                                     # m/s, Beaufort ≤4/5/6/7/8/9+

def _bl(pre, lo, hi):
    """column names cannot contain dots, else the formula parser treats them as attribute access"""
    return f"{pre}{lo}_{'p' if np.isinf(hi) else hi}".replace('.', 'p')

WV_COLS  = [_bl('Wv_', *b) for b in WAVE_BINS]
WD_COLS  = [_bl('Wd_', *b) for b in WIND_BINS]
WV_EDGES = [WAVE_BINS[0][0]] + [hi for _, hi in WAVE_BINS]
WD_EDGES = [WIND_BINS[0][0]] + [hi for _, hi in WIND_BINS]
WV_REF, WD_REF = WV_COLS[0], WD_COLS[0]                  # reference bins, must be dropped in the model
WV_REG = [c for c in WV_COLS if c != WV_REF]
WD_REG = [c for c in WD_COLS if c != WD_REF]

def build_bins(region):
    cp = cpath(region, f'bins_{BINS_VER}')
    if os.path.exists(cp): return pd.read_parquet(cp)

    wave = build_daily_cellmax(region, 'wave').rename(columns={'dmax': 'wave'})
    wind = build_daily_cellmax(region, 'wind').rename(columns={'dmax': 'wind'})
    d = wave.merge(wind, on=['grid_id', 'date'], how='inner')

    # ★ identical filtering to CELL 7 so n_days match on both sides
    n0 = len(d)
    d = d[np.isfinite(d.wave) & np.isfinite(d.wind)].copy()
    if len(d) < n0: print(f'  [{region}] dropped {n0-len(d):,} missing cell-days')

    d['year_month'] = pd.to_datetime(d['date']).dt.strftime('%Y-%m')
    d['wv'] = pd.cut(d.wave, bins=WV_EDGES, right=False, labels=WV_COLS)
    d['wd'] = pd.cut(d.wind, bins=WD_EDGES, right=False, labels=WD_COLS)
    nbad = int(d.wv.isna().sum() + d.wd.isna().sum())
    if nbad: print(f'  ⚠ [{region}] {nbad} cell-days not binned (out of range), excluded')
    d = d.dropna(subset=['wv', 'wd'])

    k  = ['grid_id', 'year_month']
    nd = d.groupby(k, observed=True).size().rename('n_days')      # ★ compute nd first
    # ★ fix ②: value_counts on Categorical takes the Cartesian product of group keys, creating
    #           ghost (grid_id, year_month) rows absent from the data. Align on nd.index to drop them.
    A = (d.groupby(k, observed=True)['wv'].value_counts()
           .unstack(fill_value=0).reindex(columns=WV_COLS, fill_value=0)
           .reindex(nd.index, fill_value=0))
    B = (d.groupby(k, observed=True)['wd'].value_counts()
           .unstack(fill_value=0).reindex(columns=WD_COLS, fill_value=0)
           .reindex(nd.index, fill_value=0))

    agg = pd.concat([nd, A, B], axis=1).reset_index()
    assert agg.isna().sum().sum() == 0, f'{region} has NaN, index misaligned'
    agg[WV_COLS + WD_COLS + ['n_days']] = agg[WV_COLS + WD_COLS + ['n_days']].astype('int16')
    assert (agg[WV_COLS].sum(1) == agg.n_days).all(), f'{region} wave bins do not close'
    assert (agg[WD_COLS].sum(1) == agg.n_days).all(), f'{region} wind bins do not close'

    agg['region'] = region
    agg = agg.merge(sea_cells[['grid_id', 'subregion']], on='grid_id', how='left')
    agg.to_parquet(cp, index=False)
    print(f'  [{region}] bins {len(agg):,} rows')
    return agg

Bins_all = pd.concat([build_bins(r) for r in REGIONS], ignore_index=True)
Bins_all.to_parquet(os.path.join(CACHE, f'bins_global_{BINS_VER}.parquet'), index=False)
print(f'\nbins: {Bins_all.shape} | total cell-days {int(Bins_all.n_days.sum()):,}')

tot = Bins_all.n_days.sum()
share = pd.DataFrame({'col': WV_COLS + WD_COLS,
                      'days': [int(Bins_all[c].sum()) for c in WV_COLS + WD_COLS]})
share['pct'] = (share.days / tot * 100).round(3)
share['ref'] = np.where(share.col.isin([WV_REF, WD_REF]), '← reference bin', '')
print('\nshare of days by bin:'); print(share.to_string(index=False))

sub_pos = (Bins_all.groupby('subregion')[WV_COLS + WD_COLS].sum() > 0).sum()
nsub = Bins_all.subregion.nunique()
print(f'\nidentifying variation by bin (sub-regions with nonzero obs / of {nsub}):')
print(sub_pos.to_string())
thin = sub_pos[sub_pos < nsub * 0.5].index.tolist()
if thin: print(f'⚠ bins observed in fewer than half the sub-regions, coefficients identified by few regions: {thin}')
print(f'\nregression columns (reference bins {WV_REF} / {WD_REF} dropped):\n  {WV_REG}\n  {WD_REG}')


In [ ]:
# ============================================================
# CELL 9 — assemble panel
#   changes: ① fix subregion column clash  ② full bookkeeping, sample loss traceable
#         ③ keep n_days / E_zero  ④ key uniqueness asserts  ⑤ sparsity diagnostics  ⑥ reduce memory
# ============================================================
KEY = ['region', 'grid_id', 'year_month']

# --- uniqueness asserts: duplicate keys in any table inflate panel rows ---
for nm, t in [('Eit', Eit_all), ('Extr', Extr_all), ('Bins', Bins_all), ('Ait', Ait)]:
    d = t.duplicated(KEY).sum()
    assert d == 0, f'{nm} has {d} duplicate-key rows, merge would inflate'

# --- bookkeeping start ---
n_acc_total = int(Ait.A_it.sum())
n_eit_all   = len(Eit_all)

panel = Eit_all[Eit_all.E_it.notna()].copy()
print(f'① E_it non-null: {len(panel):,} / {n_eit_all:,} cell-months  '
      f'(dropped {n_eit_all-len(panel):,}, no GMTDS tile coverage)')

n0 = len(panel)
panel = panel.merge(Extr_all, on=KEY, how='inner')          # Extr carries subregion, keep it
print(f'② merge Extr (wind/wave extreme days): {len(panel):,}  (dropped {n0-len(panel):,}, no ERA5 for cell-month)')

n0 = len(panel)
panel = panel.merge(Bins_all.drop(columns=['subregion']),   # ← key: drop duplicate column, else _x/_y
                    on=KEY, how='inner')
print(f'③ merge Bins (binned days): {len(panel):,}  (dropped {n0-len(panel):,})')
assert 'subregion' in panel.columns and not any(c.endswith(('_x','_y')) for c in panel.columns), \
       f'column name clash: {[c for c in panel.columns if c.endswith(("_x","_y"))]}'

panel = panel.merge(Ait.drop(columns=['subregion']), on=KEY, how='left')

# --- accident-loss bookkeeping: how many accidents did not enter the panel ---
n_acc_in = int(panel[CNT_COLS[0]].fillna(0).sum())
print(f'④ accidents in panel: {n_acc_in:,} / {n_acc_total:,} '
      f'(dropped {n_acc_total-n_acc_in:,} = {(n_acc_total-n_acc_in)/n_acc_total*100:.1f}%, '
      f'in cell-months without traffic or ERA5)')

# --- zero-fill + reduce memory ---
panel[CNT_COLS] = panel[CNT_COLS].fillna(0)
_float_cols = ['dwt_sum', 'gt_sum']
for c in CNT_COLS:
    panel[c] = panel[c].astype('float32' if c in _float_cols else 'int32')
for c in WV_COLS + WD_COLS + ['n_days']:
    panel[c] = panel[c].astype('int16')

# --- derived columns ---
panel['log_E']     = np.log(panel['E_it'] + TRAFFIC_FLOOR)
panel['E_zero']    = (panel['E_it'] == 0).astype('int8')     # true zero traffic within footprint
panel = panel.merge(sea_cells[['grid_id','grid_lat','grid_lon']], on='grid_id', how='left')
panel['year']      = panel['year_month'].str[:4].astype('int16')
panel['month']     = panel['year_month'].str[-2:]            # string, used for FE
panel['month_num'] = panel['month'].astype('int8')           # numeric, used for season interactions

outp = os.path.join(CACHE, 'panel_global.parquet')
panel.to_parquet(outp, index=False)
print(f'\n✅ panel {panel.shape} -> {outp}')
print(f'   memory {panel.memory_usage(deep=True).sum()/1e6:.0f} MB | '
      f'cell-months with E_it==0 {int(panel.E_zero.sum()):,} ({panel.E_zero.mean()*100:.1f}%)')

# --- structure table ---
print('\npanel structure by sub-region:')
print(panel.groupby(['region','subregion'])
           .agg(obs=('A_it','size'), cells=('grid_id','nunique'), months=('year_month','nunique'),
                acc=('A_it','sum'), wx=('A_wx','sum'), serious=('A_serious','sum'),
                E_med=('E_it','median'))
           .round(3).sort_values('obs', ascending=False).to_string())

# --- sparsity diagnostics: which y can support grid FE ---
YS = ['A_it','A_wx','A_placebo','A_Grounding','A_Collision','A_Contact','A_Foundered',
      'A_Hull_Damage','A_serious','A_wx_serious','A_tloss','A_poll','A_fatal']
ncell = panel.grid_id.nunique()
rows = []
for y in YS:
    pos_cells = int((panel.groupby('grid_id')[y].sum() > 0).sum())
    rows.append(dict(y=y, events=int(panel[y].sum()),
                     pos_rows_pct=round((panel[y] > 0).mean()*100, 2),
                     cells_with_events=pos_cells,
                     cells_pct=round(pos_cells/ncell*100, 1)))
diag = pd.DataFrame(rows).sort_values('events', ascending=False)
diag['grid_FE_usable'] = np.where(diag.cells_with_events >= 100, '✓',
                        np.where(diag.cells_with_events >= 30, 'marginal', '✗ fall back to sub-region/year level'))
print(f'\ndependent-variable sparsity ({ncell} cells in total):')
print(diag.to_string(index=False))
print('\nshare of zero-accident cell-months:', round((panel.A_it == 0).mean()*100, 1), '%')
print(f'regression columns: WV_REG={len(WV_REG)}  WD_REG={len(WD_REG)}  '
      f'extreme={[c for c in panel if c.endswith("_P95")]}')


In [ ]:
# ═════════════════ added at end of M2_merge: accident-level analysis dataset ═════════════════
import numpy as np, pandas as pd

XLS_FP = rf'{DATA_ROOT}/AIS_processing/accident_data/2002-2022maritime accident data.xlsx'
OUT_FP = rf'{CACHE}\M2_accident_level.parquet'
YEARS  = (2011, 2021)

panel   = pd.read_parquet(rf'{CACHE}\panel_global.parquet')
CELL2RS = panel.drop_duplicates('grid_id').set_index('grid_id')[['region','subregion']]
PANEL_KEY = set(zip(panel['grid_id'], panel['year_month']))

# ── 1. cell assignment: same rules as panel, region from panel mapping, not recomputed ──
a = pd.read_parquet(ACC_PARQUET).dropna(subset=['lat','lon'])
a = a[a['year'].between(*YEARS)].copy()
_cl = np.round(np.floor(a['lat'].values / GRID) * GRID + GRID/2, 3)
_co = np.round(np.floor(a['lon'].values / GRID) * GRID + GRID/2, 3)
a['grid_id'] = [f'{x:.3f}_{y:.3f}' for x, y in zip(_cl, _co)]
n_raw = len(a)
a = a.join(CELL2RS, on='grid_id')
a = a[a['region'].notna()].copy()
a['date']       = pd.to_datetime(a['casualty_date']).dt.normalize()
a['year_month'] = a['date'].dt.strftime('%Y-%m')
a['cmonth']     = a['date'].dt.month
a['in_panel']   = [(g, m) in PANEL_KEY for g, m in zip(a['grid_id'], a['year_month'])]

# ── 2. accident type: reuse the CELL 4 rules ──
a['atype'] = label_acc_type(a)

# ── 3. same-day wind and waves ──
_fr = []
for r in panel['region'].unique():
    w = pd.read_parquet(rf'{CACHE}\{r}__dmax_wave.parquet').rename(columns={'dmax':'swh'})
    i = pd.read_parquet(rf'{CACHE}\{r}__dmax_wind.parquet').rename(columns={'dmax':'wsp'})
    _fr.append(w.merge(i, on=['grid_id','date'], how='inner'))
DM = pd.concat(_fr, ignore_index=True).drop_duplicates(['grid_id','date'])
DM = DM[np.isfinite(DM['swh']) & np.isfinite(DM['wsp'])].copy()
DM['date'] = pd.to_datetime(DM['date'])
a = a.merge(DM[['grid_id','date','swh','wsp']], on=['grid_id','date'], how='left')
a['wx_ok'] = a['swh'].notna() & a['wsp'].notna()

# ── 4. thresholds: read those saved by CELL 7, never recompute ──
TH = pd.concat([pd.read_parquet(rf'{CACHE}\{r}__thresholds.parquet')
                for r in panel['region'].unique()], ignore_index=True)
TH = TH.pivot_table(index=['subregion','cmonth'], columns='pct',
                    values=['thr_wave','thr_wind'], aggfunc='first')
TH.columns = [f'{v}_{p}' for v, p in TH.columns]
a = a.merge(TH.reset_index(), on=['subregion','cmonth'], how='left')

for tag in ('P95','P99'):
    ew = a['swh'] >= a[f'thr_wave_{tag}']
    ei = a['wsp'] >= a[f'thr_wind_{tag}']
    a[f'haz_{tag}'] = np.where(~a['wx_ok'], None,
                       np.select([ew & ei, ei & ~ew, ew & ~ei],
                                 ['Compound','Wind only','Wave only'], 'Calm'))
    a[f'cmp_{tag}'] = (a[f'haz_{tag}'] == 'Compound').astype(float).where(a['wx_ok'])
    a[f'exc_wave_{tag}'] = (a['swh'] / a[f'thr_wave_{tag}'] - 1).clip(lower=0)   # exceedance intensity
    a[f'exc_wind_{tag}'] = (a['wsp'] / a[f'thr_wind_{tag}'] - 1).clip(lower=0)

# ── 5. consequences ──
a['serious']    = a['serious'].astype(int)
a['tloss']      = (a['loss_type'] == 'Actual total loss').astype(int)
a['ctloss']     = (a['loss_type'] == 'Constructive total loss').astype(int)
a['pollution']  = a['pollution'].astype(int)
a['n_lifelost'] = a[['n_dead','n_missing']].sum(axis=1, min_count=1)

# ── 6. ship attributes: re-attached from raw Excel (clean parquet lacks these columns) ──
raw = pd.read_excel(XLS_FP, sheet_name='data')
raw['date'] = pd.to_datetime(raw['Casualty Date'], errors='coerce').dt.normalize()
raw['age']  = pd.to_numeric(raw['Age'], errors='coerce')
V = (raw[raw['IMO'].notna() & raw['date'].notna()]
     .rename(columns={'IMO':'imo', 'Class':'cls', 'Beneficial Owner':'owner',
                      'Technical Manager':'tech_mgr'})
     [['imo','date','age','cls','owner','tech_mgr']]
     .drop_duplicates(['imo','date']))
a = a.merge(V, on=['imo','date'], how='left')

a['age_'] = a['age'].where(a['age'].between(0, 60))         # >60 treated as entry error
a['agec'] = pd.cut(a['age_'], [-.1, 5, 15, 25, 60], labels=['0-5','6-15','16-25','>25'])
a['DWT_'] = a['DWT'].replace(0, np.nan)
a['szc']  = pd.cut(a['DWT_'], [0, 5e3, 2e4, 6e4, np.inf], labels=['<5k','5-20k','20-60k','>=60k'])
_vt = a['vessel_type'].fillna('')
a['vt'] = np.select([_vt.str.contains('bulk'), _vt.str.contains('cellular|container'),
                     _vt.str.contains('oll'), _vt.str.contains('general cargo')],
                    ['Bulk','Container','RoRo','GenCargo'], 'Other')
IACS = (r'american bureau|bureau veritas|china classification|croatian register|det norske|'
        r'dnv|germanischer|indian register|korean register|lloyds register|nippon kaiji|'
        r'polski rejestr|rina|russian register')
a['cls_iacs'] = np.where(a['cls'].isna(), np.nan,
                         a['cls'].fillna('').str.lower().str.contains(IACS, regex=True).astype(float))

# ── 7. reconciliation (do not proceed if it fails) ──
n_in, n_p = int(a['in_panel'].sum()), int(panel['A_it'].sum())
print(f'cell assignment: {n_raw:,} → in regions {len(a):,} → with weather {int(a["wx_ok"].sum()):,} → in panel {n_in:,}')
assert n_in == n_p, f'✗ accident table in_panel={n_in:,} does not match panel A_it={n_p:,}'
for t in sorted(a.loc[a['in_panel'], 'atype'].unique()):
    pv = int(panel[f'A_{t}'].sum()); av = int((a['in_panel'] & (a['atype'] == t)).sum())
    assert pv == av, f'✗ {t}: panel {pv} vs accident table {av}'
print(f'✓ reconciliation passed: A_it and all 9 types match')
print(f'✓ ship age attached {a["age"].notna().sum():,}/{len(a):,} ({a["age"].notna().mean()*100:.1f}%)')

KEEP = (['imo','vessel_name','grid_id','lat','lon','date','year','year_month','cmonth',
         'region','subregion','in_panel','wx_ok','atype','cause','cause_subcategory','swh','wsp']
        + [c for c in a.columns if c.startswith(('thr_','haz_','cmp_','exc_'))]
        + ['serious','tloss','ctloss','pollution','n_dead','n_injured','n_missing','n_lifelost',
           'age','age_','agec','DWT','DWT_','GT','szc','vessel_type','vt','flag',
           'cls','cls_iacs','owner','tech_mgr'])
a[[c for c in KEEP if c in a.columns]].to_parquet(OUT_FP, index=False)
print(f'written {OUT_FP}  ({len(a):,} rows × {len([c for c in KEEP if c in a.columns])} cols)')


In [ ]:
shutil.rmtree(CACHE, ignore_errors=True); os.makedirs(CACHE, exist_ok=True)
print('cache cleared:', CACHE)


In [ ]:
mani = pd.read_csv(os.path.join(GMTDS_ORG, '_manifest.csv'))
mani.columns = [c.strip().lstrip('\ufeff') for c in mani.columns]
box2regs = {int(r.box_id): str(r.regions).split(';') for r in mani.itertuples()}
pat = re.compile(r'Cargo_b(\d+)_(\d{4})_(\d{1,2})\.tif$', re.I)
rows = []
for p in glob.glob(os.path.join(GMTDS_ORG, '**', '*.tif'), recursive=True):
    m = pat.search(os.path.basename(p))
    if m and int(m.group(2)) in YEARS:
        rows.append((int(m.group(1)), int(m.group(2)), int(m.group(3)), p))
tif_idx = pd.DataFrame(rows, columns=['box_id','year','month','path']).drop_duplicates(['box_id','year','month'])
def boxes_of_region(region): return [b for b, rr in box2regs.items() if region in rr]
print('CELL2 OK | tif entries', len(tif_idx))


In [ ]:
def build_sea_cells():
    cp = os.path.join(CACHE, '_sea_cells.parquet')
    if os.path.exists(cp): return pd.read_parquet(cp)
    ds = xr.open_dataset(os.path.join(WAVE_DIR, f'era5_swh_{YEARS[0]}.nc'))
    la = ds['latitude'].values; lo = ds['longitude'].values
    lo180 = np.where(lo > 180, lo-360, lo)
    ocean = ~np.isnan(ds['swh'].isel(valid_time=0).values); ds.close()
    recs = []
    for i, a in enumerate(la):
        ro = ocean[i]
        for j, o in enumerate(lo180):
            if not ro[j]: continue
            reg = assign_region(a, o)
            if reg is None: continue
            clat, clon = to_cell(a, o)
            recs.append((reg, gid(clat, clon), clat, clon))
    sea = pd.DataFrame(recs, columns=['region','grid_id','grid_lat','grid_lon']).drop_duplicates('grid_id').reset_index(drop=True)
    sea.to_parquet(cp, index=False); return sea

sea_cells = build_sea_cells()
print(sea_cells.region.value_counts().to_string())
fig, ax = plt.subplots(figsize=(13,6)); colors = dict(zip(REGIONS, plt.cm.tab10.colors))
for r, g in sea_cells.groupby('region'):
    ax.scatter(g.grid_lon, g.grid_lat, s=8, color=colors[r], label=r)
for r, boxes in REGIONS.items():
    for (S,N,W,E) in boxes:
        ax.add_patch(plt.Rectangle((W,S), E-W, N-S, fill=False, edgecolor=colors[r], lw=1.4, ls='--'))
ax.legend(loc='lower left', ncol=3, fontsize=8); ax.grid(alpha=.3)
ax.set_title(f'sea cells (grid={GRID_SIZE}) n={len(sea_cells)}'); plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# CELL 4b — accident locations by region (with land); green=in panel red×=dropped on land cell
# ============================================================
import cartopy.crs as ccrs, cartopy.feature as cfeature

_acc = pd.read_parquet(ACC_PARQUET).dropna(subset=['lat','lon'])
_acc = _acc[_acc.year.between(YEARS[0], YEARS[-1])].copy()
_acc['region'] = [assign_region(a, o) for a, o in zip(_acc.lat, _acc.lon)]
_acc = _acc[_acc.region.notna()].copy()
_clat, _clon = to_cell(_acc.lat.values, _acc.lon.values)
_acc['grid_id'] = [gid(a, o) for a, o in zip(_clat, _clon)]
_acc['in_sea']  = _acc.grid_id.isin(set(sea_cells.grid_id))

fig, axes = plt.subplots(2, 3, figsize=(19, 11), subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (r, boxes) in zip(axes.ravel(), REGIONS.items()):
    S=min(b[0] for b in boxes); N=max(b[1] for b in boxes)
    W=min(b[2] for b in boxes); E=max(b[3] for b in boxes)
    mg=5; ax.set_extent([W-mg, E+mg, S-mg, N+mg], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#e4d9c3'); ax.add_feature(cfeature.OCEAN, facecolor='#d6e6ef')
    ax.add_feature(cfeature.COASTLINE, lw=.5)
    g  = _acc[_acc.region == r]
    gk, gd = g[g.in_sea], g[~g.in_sea]
    ax.scatter(gk.lon, gk.lat, s=9,  c='green', alpha=.45, transform=ccrs.PlateCarree(), zorder=4, label=f'in panel {len(gk)}')
    ax.scatter(gd.lon, gd.lat, s=14, c='red', marker='x', alpha=.7, transform=ccrs.PlateCarree(), zorder=5, label=f'dropped (land cell) {len(gd)}')
    for (s, n, w, e) in boxes:
        ax.add_patch(plt.Rectangle((w, s), e-w, n-s, transform=ccrs.PlateCarree(),
                     fill=False, edgecolor='navy', lw=1.6, zorder=6))
    glx = ax.gridlines(draw_labels=True, lw=.3, alpha=.3); glx.top_labels=glx.right_labels=False
    glx.xlabel_style = glx.ylabel_style = {'size':7}
    ax.set_title(f'{r}  (accidents {len(g)})', fontsize=10); ax.legend(loc='lower left', fontsize=7)
plt.suptitle('Accident locations by region · green=in panel · red×=dropped on land cell · blue box=region bounds', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
def build_Ait():
    cp = os.path.join(CACHE, '_Ait.parquet')
    if os.path.exists(cp): return pd.read_parquet(cp)
    df = pd.read_parquet(ACC_PARQUET).dropna(subset=['lat','lon'])
    df = df[df.year.between(YEARS[0], YEARS[-1])].copy()
    df['region'] = [assign_region(a,o) for a,o in zip(df.lat, df.lon)]
    df = df[df.region.notna()].copy()
    clat, clon = to_cell(df.lat.values, df.lon.values)
    df['grid_id'] = [gid(a,o) for a,o in zip(clat,clon)]
    df = df[df.grid_id.isin(set(sea_cells.grid_id))].copy()
    df['year_month'] = pd.to_datetime(df['casualty_date']).dt.strftime('%Y-%m')
    keys = ['region','grid_id','year_month']
    Ait = df.groupby(keys).size().rename('A_it').reset_index()
    cause = (df['cause'].fillna('').astype(str) + ' ' +
             df['cause_subcategory'].fillna('').astype(str)).str.lower()
    for t, rgx in TYPE_FILTERS.items():
        c = df[cause.str.contains(rgx, regex=True, na=False)].groupby(keys).size().rename(f'A_{t}').reset_index()
        Ait = Ait.merge(c, on=keys, how='left')
    tcols = [f'A_{t}' for t in TYPE_FILTERS]
    Ait[tcols] = Ait[tcols].fillna(0).astype(int)
    Ait.to_parquet(cp, index=False); return Ait

Ait = build_Ait()
print(f'A_it {len(Ait)} rows | total accidents={Ait.A_it.sum()}')
print(Ait.groupby('region').A_it.sum().to_string())
print('totals by type:', {t: int(Ait[f'A_{t}'].sum()) for t in TYPE_FILTERS})


In [ ]:
def _bin_tif_to_cells(path, G=GRID_SIZE, dec=DECIMATE):
    with rasterio.open(path) as src:
        oh, ow = max(1, src.height//dec), max(1, src.width//dec)
        arr = src.read(1, out_shape=(oh, ow), resampling=Resampling.average, masked=True)
        T = src.transform * src.transform.scale(src.width/ow, src.height/oh)
    valid = ~arr.mask if np.ma.isMaskedArray(arr) else np.isfinite(arr)
    data  = arr.data if np.ma.isMaskedArray(arr) else arr
    ny, nx = arr.shape
    xs = T.c + (np.arange(nx)+0.5)*T.a; ys = T.f + (np.arange(ny)+0.5)*T.e
    clon = np.floor(xs/G)*G+G/2; clat = np.floor(ys/G)*G+G/2
    d = pd.DataFrame({'grid_lat': np.round(np.broadcast_to(clat[:,None], arr.shape)[valid],3),
                      'grid_lon': np.round(np.broadcast_to(clon, arr.shape)[valid],3),
                      'v': data[valid]})
    g = d.groupby(['grid_lat','grid_lon'])['v'].agg(psum='sum', pcount='count').reset_index()
    g['grid_id'] = [gid(a,o) for a,o in zip(g.grid_lat, g.grid_lon)]; return g

def build_Eit(region):
    cp = cpath(region, 'Eit')
    if os.path.exists(cp): return pd.read_parquet(cp)
    sea_ids = set(sea_cells[sea_cells.region==region].grid_id)
    sub = tif_idx[tif_idx.box_id.isin(boxes_of_region(region))]
    out = []
    for (y, mo), grp in sub.groupby(['year','month']):
        acc = {}
        for p in grp.path:
            try: g = _bin_tif_to_cells(p)
            except Exception as e: print('skip', os.path.basename(p), e); continue
            for r in g.itertuples(index=False):
                if r.grid_id not in sea_ids: continue
                s, c = acc.get(r.grid_id, (0.0, 0)); acc[r.grid_id] = (s+r.psum, c+r.pcount)
        for k, (s, c) in acc.items():
            out.append((region, k, f'{y}-{mo:02d}', s/c if c else np.nan))
    Eit = pd.DataFrame(out, columns=['region','grid_id','year_month','E_it'])
    Eit.to_parquet(cp, index=False); print(f'  [{region}] E_it {len(Eit)}'); return Eit

Eit_all = pd.concat([build_Eit(r) for r in REGIONS], ignore_index=True)
nmax = len(YEARS)*12
cov = (Eit_all[Eit_all.E_it.notna()].groupby(['region','grid_id']).size().rename('n').reset_index())
cov = sea_cells.merge(cov, on=['region','grid_id'], how='left').fillna({'n':0}); cov['frac']=cov.n/nmax
print(cov.groupby('region').frac.describe()[['mean','min','25%','50%']].round(2).to_string())
fig, ax = plt.subplots(figsize=(13,5))
sc = ax.scatter(cov.grid_lon, cov.grid_lat, c=cov.frac, s=14, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(sc, ax=ax, shrink=.8); draw_region_boxes(ax); ax.set_title('traffic monthly coverage')
plt.tight_layout(); plt.show()


In [ ]:
def _open_var(year, var):
    if var == 'wave':
        ds = xr.open_dataset(os.path.join(WAVE_DIR, f'era5_swh_{year}.nc'), chunks={'valid_time':240})
        da = ds['swh']
    else:
        ds = xr.open_dataset(os.path.join(WIND_DIR, f'era5_wind10m_{year}.grib'),
                             engine='netcdf4', chunks={'valid_time':240})
        da = np.sqrt(ds['u10']**2 + ds['v10']**2)
    da = da.squeeze(drop=True)
    da = da.assign_coords(longitude=(((da['longitude']+180)%360)-180)).sortby('longitude')
    return ds, da

def build_daily_cellmax(region, var):
    cp = cpath(region, f'dmax_{var}')
    if os.path.exists(cp): return pd.read_parquet(cp)
    S,N,W,E = region_bounds(region); sea_ids = set(sea_cells[sea_cells.region==region].grid_id)
    frames = []
    for y in YEARS:
        ds, da = _open_var(y, var)
        da = da.sel(latitude=slice(N,S), longitude=slice(W,E))
        dd = da.resample(valid_time='1D').max().load()
        clat = np.round(np.floor(dd.latitude.values/GRID_SIZE)*GRID_SIZE+GRID_SIZE/2, 3)
        clon = np.round(np.floor(dd.longitude.values/GRID_SIZE)*GRID_SIZE+GRID_SIZE/2, 3)
        dd = dd.assign_coords(clat=('latitude',clat), clon=('longitude',clon))
        cell = dd.groupby('clat').max().groupby('clon').max()
        df = cell.to_dataframe(name='dmax').reset_index()
        df['grid_id'] = df.clat.map('{:.3f}'.format) + '_' + df.clon.map('{:.3f}'.format)
        df = df[df.grid_id.isin(sea_ids)].copy()
        df['date'] = pd.to_datetime(df['valid_time']).dt.normalize()
        frames.append(df[['grid_id','date','dmax']]); ds.close(); print(f'  {region}/{var} {y}', end=' ')
    out = pd.concat(frames, ignore_index=True); out.to_parquet(cp, index=False); print(); return out
print('CELL6 definitions done')

In [ ]:
PCTS = [('P95', 0.95), ('P99', 0.99)]
def build_extrdays(region):
    cp = cpath(region, 'extrdays')
    if os.path.exists(cp): return pd.read_parquet(cp)
    wave = build_daily_cellmax(region, 'wave').rename(columns={'dmax':'wave'})
    wind = build_daily_cellmax(region, 'wind').rename(columns={'dmax':'wind'})
    d = wave.merge(wind, on=['grid_id','date'], how='inner')
    d['year_month'] = pd.to_datetime(d['date']).dt.strftime('%Y-%m')
    ccols = []
    for tag, q in PCTS:
        tw = d.wave.quantile(q); ti = d.wind.quantile(q)     # single regional annual threshold
        ew = d.wave >= tw; ei = d.wind >= ti
        d[f'wind_only_{tag}']    = (ei & ~ew).astype('int8')
        d[f'wave_only_{tag}']    = (ew & ~ei).astype('int8')
        d[f'compound_{tag}']     = (ew & ei).astype('int8')
        d[f'wind_or_wave_{tag}'] = (ew | ei).astype('int8')
        ccols += [f'wind_only_{tag}', f'wave_only_{tag}', f'compound_{tag}', f'wind_or_wave_{tag}']
    agg = d.groupby(['grid_id','year_month'])[ccols].sum().reset_index()
    agg['region'] = region; agg.to_parquet(cp, index=False)
    print(f'  [{region}] extrdays {len(agg)}'); return agg

Extr_all = pd.concat([build_extrdays(r) for r in REGIONS], ignore_index=True)
print('total days by class (P95):')
print(Extr_all[[f'{c}_P95' for c in ['wind_only','wave_only','compound','wind_or_wave']]].sum().to_string())


In [ ]:
WAVE_BINS = [(0,2),(2,3),(3,4),(4,5),(5,6),(6,np.inf)]          # m
WIND_BINS = [(0,10),(10,14),(14,17),(17,20),(20,24),(24,np.inf)] # m/s
def _bl(pre, lo, hi): return f"{pre}{lo}_{'p' if np.isinf(hi) else hi}"

def build_bins(region):
    cp = cpath(region, 'bins')
    if os.path.exists(cp): return pd.read_parquet(cp)
    wave = build_daily_cellmax(region, 'wave').rename(columns={'dmax':'wave'})
    wind = build_daily_cellmax(region, 'wind').rename(columns={'dmax':'wind'})
    d = wave.merge(wind, on=['grid_id','date'], how='inner')
    d['year_month'] = pd.to_datetime(d['date']).dt.strftime('%Y-%m')
    for lo,hi in WAVE_BINS: d[_bl('Wv_',lo,hi)] = ((d.wave>=lo)&(d.wave<hi)).astype('int8')
    for lo,hi in WIND_BINS: d[_bl('Wd_',lo,hi)] = ((d.wind>=lo)&(d.wind<hi)).astype('int8')
    cols = [_bl('Wv_',*b) for b in WAVE_BINS] + [_bl('Wd_',*b) for b in WIND_BINS]
    agg = d.groupby(['grid_id','year_month'])[cols].sum().reset_index()
    agg['region'] = region; agg.to_parquet(cp, index=False); return agg

Bins_all = pd.concat([build_bins(r) for r in REGIONS], ignore_index=True)
Bins_all.to_parquet(os.path.join(CACHE, 'bins_global.parquet'), index=False)
print('bins:', Bins_all.shape)
print('total days per wave bin:', {c:int(Bins_all[c].sum()) for c in Bins_all if c.startswith('Wv_')})
print('total days per wind bin:', {c:int(Bins_all[c].sum()) for c in Bins_all if c.startswith('Wd_')})


In [ ]:
acols = ['A_it'] + [f'A_{t}' for t in TYPE_FILTERS]
panel = Eit_all[Eit_all.E_it.notna()].copy()
panel = panel.merge(Extr_all, on=['region','grid_id','year_month'], how='inner')
panel = panel.merge(Ait, on=['region','grid_id','year_month'], how='left')
panel[acols] = panel[acols].fillna(0).astype(int)
panel['log_E'] = np.log(panel['E_it'] + TRAFFIC_FLOOR)
panel = panel.merge(sea_cells[['grid_id','grid_lat','grid_lon']], on='grid_id', how='left')
panel['month'] = panel['year_month'].str[-2:]
outp = os.path.join(CACHE, 'panel_global.parquet'); panel.to_parquet(outp, index=False)
print(f'✅ panel {panel.shape} -> {outp}')
print(panel.groupby('region').agg(obs=('A_it','size'), cells=('grid_id','nunique'), acc=('A_it','sum')).to_string())


In [ ]:
# ============================================================
# M2 panel · CELL 1 — config
# ============================================================
import os, re, glob, warnings
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.enums import Resampling
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────
ACC_PARQUET = rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet'
GMTDS_ORG   = rf'{DATA_ROOT}/climate_AIS_data/GMTDS/cargo_ship/organized'
WAVE_DIR    = rf'{DATA_ROOT}/AIS_processing/weather_data/wave'   # era5_swh_{y}.nc
WIND_DIR    = rf'{DATA_ROOT}/AIS_processing/weather_data/wind'   # era5_wind10m_{y}.grib (actually NetCDF)
CACHE_BASE  = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2'

# ── key parameters ──────────────────────────────────────────────
GRID_SIZE = 2.0                       # main 2.0; rerun with 1.0 for sensitivity
YEARS     = list(range(2011, 2022))   # 2011-2021
DECIMATE  = 4                         # downsampling factor when reading GMTDS rasters (2x2 mean suffices, ~16x faster)
TRAFFIC_FLOOR = 0.5                   # use log(E+FLOOR) as floor when E=0

# ── 6 non-overlapping region bounds (W, E, S, N), unique assignment by cell centre ──────────
# ── revised 6 region bounds (S, N, W, E), unique assignment by cell centre ──
REGIONS = {
    "NW_Europe":   [(36, 63, -15, -6), (43, 63, -6, 0), (48, 63, 0, 13)],
    "NW_Atlantic": [(18, 50, -98, -60)],
    "SE_Asia":     [(-10, 20, 95, 127)],
    "E_Asia":      [(20, 42, 117, 145)],
    "Med_Black":   [(30, 43, -6, 0), (30, 47.5, 0, 42)],
    "Baltic":      [(53, 66, 13, 30)],
}

CACHE = os.path.join(CACHE_BASE, f'grid{GRID_SIZE:g}')
os.makedirs(CACHE, exist_ok=True)
def cpath(region, name): return os.path.join(CACHE, f'{region}__{name}.parquet')

# ── grid/region helpers ────────────────────────────────────────
def to_cell(lat, lon, G=GRID_SIZE):
    """lat/lon -> grid cell centre (clat, clon)"""
    clat = np.floor(lat / G) * G + G / 2
    clon = np.floor(lon / G) * G + G / 2
    return np.round(clat, 3), np.round(clon, 3)

def gid(clat, clon):
    return f'{clat:.3f}_{clon:.3f}'
def assign_region(lat, lon):
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            if (S <= lat < N) and (W <= lon < E):
                return r
    return None

def draw_region_boxes(ax, color='k', **kw):
    st = dict(fill=False, edgecolor=color, lw=.8, ls='--'); st.update(kw)
    for r, boxes in REGIONS.items():
        for (S, N, W, E) in boxes:
            ax.add_patch(plt.Rectangle((W, S), E-W, N-S, **st))

print(f'CELL1 OK | GRID_SIZE={GRID_SIZE} | cache={CACHE}')


In [ ]:
# ============================================================
# CELL 2 — index GMTDS rasters + box→region mapping from manifest
# ============================================================
# manifest: box_id, W,S,E,N, n_months, regions (possibly "SE_Asia;E_Asia")
mani = pd.read_csv(os.path.join(GMTDS_ORG, '_manifest.csv'))
mani.columns = [c.strip().lstrip('\ufeff') for c in mani.columns]
box2regs = {int(r.box_id): str(r.regions).split(';') for r in mani.itertuples()}

# scan all tifs, parse (box_id, year, month, path)
pat = re.compile(r'Cargo_b(\d+)_(\d{4})_(\d{1,2})\.tif$', re.I)
rows = []
for p in glob.glob(os.path.join(GMTDS_ORG, '**', '*.tif'), recursive=True):
    m = pat.search(os.path.basename(p))
    if not m: continue
    bid, y, mo = int(m.group(1)), int(m.group(2)), int(m.group(3))
    if y in YEARS:
        rows.append((bid, y, mo, p))
tif_idx = pd.DataFrame(rows, columns=['box_id', 'year', 'month', 'path'])
# deduplicate (same box-month may be copied into several region folders)
tif_idx = tif_idx.drop_duplicates(['box_id', 'year', 'month'])

def boxes_of_region(region):
    return [b for b, regs in box2regs.items() if region in regs]

print(f'CELL2 OK | tif entries={len(tif_idx)} | unique boxes={tif_idx.box_id.nunique()}')
for r in REGIONS:
    bs = boxes_of_region(r)
    n  = tif_idx[tif_idx.box_id.isin(bs)].shape[0]
    print(f'  {r:12s}: boxes={bs}  tif months={n}')


In [ ]:
import shutil; shutil.rmtree(CACHE, ignore_errors=True); os.makedirs(CACHE, exist_ok=True)

In [ ]:
# ============================================================
# CELL 3 — sea mask (valid SWH = sea) + sea cell list per region
#          using one SWH snapshot (land=NaN is static)
# ============================================================
def build_sea_cells():
    cp = os.path.join(CACHE, '_sea_cells.parquet')
    if os.path.exists(cp):
        return pd.read_parquet(cp)
    y0 = YEARS[0]
    ds = xr.open_dataset(os.path.join(WAVE_DIR, f'era5_swh_{y0}.nc'))
    la = ds['latitude'].values
    lo = ds['longitude'].values
    lo180 = np.where(lo > 180, lo - 360, lo)        # 0-360 -> -180..180
    ocean = ~np.isnan(ds['swh'].isel(valid_time=0).values)   # True=sea
    ds.close()

    recs = []
    for i, a in enumerate(la):
        r_ocean = ocean[i]
        for j, o in enumerate(lo180):
            if not r_ocean[j]:
                continue
            reg = assign_region(a, o)
            if reg is None:
                continue
            clat, clon = to_cell(a, o)
            recs.append((reg, gid(clat, clon), clat, clon))
    sea = (pd.DataFrame(recs, columns=['region', 'grid_id', 'grid_lat', 'grid_lon'])
             .drop_duplicates('grid_id').reset_index(drop=True))
    sea.to_parquet(cp, index=False)
    return sea

sea_cells = build_sea_cells()
print('sea cells / region:')
print(sea_cells.region.value_counts().to_string())

# ── VIZ 1: region bounds + sea cells ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(13, 6))
colors = dict(zip(REGIONS, plt.cm.tab10.colors))
for r, g in sea_cells.groupby('region'):
    ax.scatter(g.grid_lon, g.grid_lat, s=8, color=colors[r], label=r)
for r, boxes in REGIONS.items():
    for (S, N, W, E) in boxes:
        ax.add_patch(plt.Rectangle((W, S), E-W, N-S, fill=False, edgecolor=colors[r], lw=1.5, ls='--'))

ax.set(xlabel='lon', ylabel='lat',
       title=f'6 regions & sea cells (grid={GRID_SIZE}°) — n={len(sea_cells)}')
ax.legend(loc='lower left', ncol=3, fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# CELL 4 — A_it: accident count per sea cell × year_month
# ============================================================
def build_Ait():
    cp = os.path.join(CACHE, '_Ait.parquet')
    if os.path.exists(cp):
        return pd.read_parquet(cp)
    df = pd.read_parquet(ACC_PARQUET).dropna(subset=['lat', 'lon'])
    df = df[df.year.between(YEARS[0], YEARS[-1])].copy()
    df['region'] = [assign_region(a, o) for a, o in zip(df.lat, df.lon)]
    df = df[df.region.notna()].copy()
    clat, clon = to_cell(df.lat.values, df.lon.values)
    df['grid_id'] = [gid(a, o) for a, o in zip(clat, clon)]
    df = df[df.grid_id.isin(set(sea_cells.grid_id))].copy()     # sea cells only
    dt = pd.to_datetime(df['casualty_date'])
    df['year_month'] = dt.dt.strftime('%Y-%m')
    Ait = (df.groupby(['region', 'grid_id', 'year_month'])
             .size().rename('A_it').reset_index())
    Ait.to_parquet(cp, index=False)
    return Ait

Ait = build_Ait()
print(f'A_it: {len(Ait)} rows | total accidents={Ait.A_it.sum()}')
print(Ait.groupby('region').A_it.sum().to_string())


In [ ]:
# ============================================================
# CELL 5 (fixed) — E_it: cargo shipping density per sea cell × month
# fix: rename aggregate columns psum/pcount to avoid itertuples name clash
# ============================================================
def _bin_tif_to_cells(path, G=GRID_SIZE, dec=DECIMATE):
    """read one tif -> DataFrame[grid_id, grid_lat, grid_lon, psum, pcount]"""
    with rasterio.open(path) as src:
        oh, ow = max(1, src.height // dec), max(1, src.width // dec)
        arr = src.read(1, out_shape=(oh, ow),
                       resampling=Resampling.average, masked=True)
        T = src.transform * src.transform.scale(src.width / ow, src.height / oh)
    if np.ma.isMaskedArray(arr):
        valid = ~arr.mask
        data  = arr.data
    else:
        valid = np.isfinite(arr); data = arr
    ny, nx = arr.shape
    xs = T.c + (np.arange(nx) + 0.5) * T.a        # lon (per column)
    ys = T.f + (np.arange(ny) + 0.5) * T.e        # lat (per row, T.e<0)
    clon = (np.floor(xs / G) * G + G / 2)
    clat = (np.floor(ys / G) * G + G / 2)
    CLON = np.broadcast_to(clon, arr.shape)[valid]
    CLAT = np.broadcast_to(clat[:, None], arr.shape)[valid]
    V    = data[valid]
    d = pd.DataFrame({'grid_lat': np.round(CLAT, 3),
                      'grid_lon': np.round(CLON, 3), 'v': V})
    g = (d.groupby(['grid_lat', 'grid_lon'])['v']
           .agg(psum='sum', pcount='count').reset_index())
    g['grid_id'] = [gid(a, o) for a, o in zip(g.grid_lat, g.grid_lon)]
    return g

def build_Eit(region):
    cp = cpath(region, 'Eit')
    if os.path.exists(cp):
        return pd.read_parquet(cp)
    sea_ids = set(sea_cells[sea_cells.region == region].grid_id)
    bs = boxes_of_region(region)
    sub = tif_idx[tif_idx.box_id.isin(bs)]
    out = []
    for (y, mo), grp in sub.groupby(['year', 'month']):
        acc = {}   # grid_id -> [sum, count]
        for p in grp.path:
            try:
                g = _bin_tif_to_cells(p)
            except Exception as e:
                print(f'    skip {os.path.basename(p)}: {e}'); continue
            for r in g.itertuples(index=False):
                if r.grid_id not in sea_ids:
                    continue
                s, c = acc.get(r.grid_id, (0.0, 0))
                acc[r.grid_id] = (s + r.psum, c + r.pcount)
        for k, (s, c) in acc.items():
            out.append((region, k, f'{y}-{mo:02d}', s / c if c else np.nan))
    Eit = pd.DataFrame(out, columns=['region', 'grid_id', 'year_month', 'E_it'])
    Eit.to_parquet(cp, index=False)
    print(f'  [{region}] E_it: {len(Eit)} cell-months')
    return Eit

# build per region (slow first time, then read from cache)
Eit_all = pd.concat([build_Eit(r) for r in REGIONS], ignore_index=True)
print(f'E_it total: {len(Eit_all)} rows')


In [ ]:
# ── coverage: months out of 132 with traffic for each sea cell ──
nmax = len(YEARS) * 12
cov = (Eit_all[Eit_all.E_it.notna()]
       .groupby(['region', 'grid_id']).size().rename('n_months').reset_index())
cov = sea_cells.merge(cov, on=['region', 'grid_id'], how='left').fillna({'n_months': 0})
cov['cover_frac'] = cov.n_months / nmax
print('\n=== sea cell coverage by region (share of months covered) ===')
print(cov.groupby('region').cover_frac.describe()[['mean', 'min', '25%', '50%']].round(2).to_string())
print('\nsea cells with no traffic at all / region:')
print(cov[cov.n_months == 0].groupby('region').size().to_string())

# ── VIZ 2: traffic coverage + mean density ──
mean_dens = (Eit_all[Eit_all.E_it > 0].groupby('grid_id').E_it.mean()
             .rename('mean_dens').reset_index())
cov2 = cov.merge(mean_dens, on='grid_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
sc0 = axes[0].scatter(cov2.grid_lon, cov2.grid_lat, c=cov2.cover_frac,
                      s=14, cmap='RdYlGn', vmin=0, vmax=1)
axes[0].set_title('traffic monthly coverage (1=all 132 months)')
plt.colorbar(sc0, ax=axes[0], shrink=.8)
sc1 = axes[1].scatter(cov2.grid_lon, cov2.grid_lat,
                      c=np.log1p(cov2.mean_dens), s=14, cmap='viridis')
axes[1].set_title('mean cargo density log1p(mean)')
plt.colorbar(sc1, ax=axes[1], shrink=.8)
for ax in axes:
    draw_region_boxes(ax, lw=.6)
    ax.set(xlabel='lon', ylabel='lat'); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# CELL 6 — daily max per sea cell × day: wave (SWH) and wind (wind speed)
#   wind speed=√(u10²+v10²); hourly first, then daily max, then max within cell
#   ERA5: lon 0-360→±180; lat descending; stream with dask chunks to limit memory
# ============================================================
def region_bounds(region):
    bs = REGIONS[region]
    return (min(b[0] for b in bs), max(b[1] for b in bs),
            min(b[2] for b in bs), max(b[3] for b in bs))

def _open_var(year, var):
    if var == 'wave':
        ds = xr.open_dataset(os.path.join(WAVE_DIR, f'era5_swh_{year}.nc'),
                             chunks={'valid_time': 240})
        da = ds['swh']
    else:
        ds = xr.open_dataset(os.path.join(WIND_DIR, f'era5_wind10m_{year}.grib'),
                             engine='netcdf4', chunks={'valid_time': 240})
        da = np.sqrt(ds['u10'] ** 2 + ds['v10'] ** 2)
    da = da.squeeze(drop=True)
    lon = da['longitude']
    da = da.assign_coords(longitude=(((lon + 180) % 360) - 180)).sortby('longitude')
    return ds, da

def build_daily_cellmax(region, var):
    cp = cpath(region, f'dmax_{var}')
    if os.path.exists(cp):
        return pd.read_parquet(cp)
    S, N, W, E = region_bounds(region)
    sea_ids = set(sea_cells[sea_cells.region == region].grid_id)
    frames = []
    for y in YEARS:
        ds, da = _open_var(y, var)
        da = da.sel(latitude=slice(N, S), longitude=slice(W, E))   # ERA5 lat descending
        dd = da.resample(valid_time='1D').max().load()             # hourly→daily max (streamed compute)
        clat = np.round(np.floor(dd.latitude.values / GRID_SIZE) * GRID_SIZE + GRID_SIZE/2, 3)
        clon = np.round(np.floor(dd.longitude.values / GRID_SIZE) * GRID_SIZE + GRID_SIZE/2, 3)
        dd = dd.assign_coords(clat=('latitude', clat), clon=('longitude', clon))
        cell = dd.groupby('clat').max().groupby('clon').max()      # max within cell
        df = cell.to_dataframe(name='dmax').reset_index()
        df['grid_id'] = df.clat.map('{:.3f}'.format) + '_' + df.clon.map('{:.3f}'.format)
        df = df[df.grid_id.isin(sea_ids)].copy()
        df['date'] = pd.to_datetime(df['valid_time']).dt.normalize()
        frames.append(df[['grid_id', 'date', 'dmax']])
        ds.close()
        print(f'    [{region}/{var}] {y} ok', end='  ')
    out = pd.concat(frames, ignore_index=True)
    out.to_parquet(cp, index=False)
    print(f'\n  [{region}/{var}] done: {len(out)} cell-days')
    return out

print('CELL6 definitions done')


In [ ]:
# ============================================================
# CELL 7 — regional P95/P99 (scalar per calendar month) → wind_only/wave_only/compound/wind_or_wave
#   extreme wave day: cell daily max SWH ≥ regional P for that calendar month; same for wind
#   compound = both exceed in same cell and day; wind_or_wave = union
# ============================================================
PCTS = [('P95', 0.95), ('P99', 0.99)]

def build_extrdays(region):
    cp = cpath(region, 'extrdays')
    if os.path.exists(cp):
        return pd.read_parquet(cp)
    wave = build_daily_cellmax(region, 'wave').rename(columns={'dmax': 'wave'})
    wind = build_daily_cellmax(region, 'wind').rename(columns={'dmax': 'wind'})
    d = wave.merge(wind, on=['grid_id', 'date'], how='inner')
    d['month'] = d.date.dt.month
    d['year_month'] = d.date.dt.strftime('%Y-%m')

    ccols = []
    for tag, q in PCTS:
        tw = d.wave.quantile(q)     # single regional threshold (2011-2021 full period), scalar
        ti = d.wind.quantile(q)
        ew = d.wave >= tw
        ei = d.wind >= ti
        d[f'wind_only_{tag}']    = (ei & ~ew).astype('int8')
        d[f'wave_only_{tag}']    = (ew & ~ei).astype('int8')
        d[f'compound_{tag}']     = (ew & ei).astype('int8')
        d[f'wind_or_wave_{tag}'] = (ew | ei).astype('int8')
        ccols += [f'wind_only_{tag}', f'wave_only_{tag}',
                  f'compound_{tag}', f'wind_or_wave_{tag}']

    agg = d.groupby(['grid_id', 'year_month'])[ccols].sum().reset_index()
    agg['region'] = region
    agg.to_parquet(cp, index=False)
    print(f'  [{region}] extrdays: {len(agg)} cell-months')
    return agg

Extr_all = pd.concat([build_extrdays(r) for r in REGIONS], ignore_index=True)
print(f'\nExtrDays total: {len(Extr_all)} rows')
print('total days by class (P95):')
print(Extr_all[['wind_only_P95','wave_only_P95','compound_P95','wind_or_wave_P95']].sum().to_string())


In [ ]:
# ============================================================
# CELL 8 — merge global panel: E_it (backbone) ⋈ ExtrDays ⋈ A_it
#   E_it no coverage (NaN)→drop row; A_it missing→fill 0; log_E=log(E+floor)
# ============================================================
panel = Eit_all[Eit_all.E_it.notna()].copy()                              # has traffic coverage
panel = panel.merge(Extr_all, on=['region','grid_id','year_month'], how='inner')   # weather (inner)
panel = panel.merge(Ait[['region','grid_id','year_month','A_it']],
                    on=['region','grid_id','year_month'], how='left')       # accidents (left)
panel['A_it'] = panel['A_it'].fillna(0).astype(int)
panel['log_E'] = np.log(panel['E_it'] + TRAFFIC_FLOOR)
panel = panel.merge(sea_cells[['grid_id','grid_lat','grid_lon']], on='grid_id', how='left')
panel['month'] = panel['year_month'].str[-2:]

outp = os.path.join(CACHE, 'panel_global.parquet')
panel.to_parquet(outp, index=False)

print(f'✅ global panel: {panel.shape[0]:,} rows × {panel.shape[1]} cols')
print(f'   saved: {outp}\n')
summ = panel.groupby('region').agg(obs=('A_it','size'), cells=('grid_id','nunique'),
                                   months=('year_month','nunique'), acc=('A_it','sum'))
print(summ.to_string())
print(f'\ntotal obs={len(panel):,} | total accidents={panel.A_it.sum():,} | '
      f'nonzero A_it share={100*(panel.A_it>0).mean():.1f}%')
print('columns:', list(panel.columns))


In [ ]:
# ── VIZ 4: panel health check ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].hist(panel.A_it[panel.A_it<=panel.A_it.quantile(.999)], bins=range(0,8), rwidth=.85)
axes[0].set(title='A_it distribution (counts)', yscale='log')
axes[1].scatter(panel.log_E, panel.compound_P95, s=4, alpha=.15)
axes[1].set(title='log_E vs compound days', xlabel='log_E', ylabel='compound_P95 (days/month)')
axes[2].scatter(panel.grid_lon, panel.grid_lat,
                c=(panel.groupby('grid_id')['A_it'].transform('sum')>0).astype(int),
                s=6, cmap='RdYlGn')
draw_region_boxes(axes[2]); axes[2].set(title='cells with accidents (green)')
for a in axes: a.grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# M2 PPML analysis · A1 — load global panel
# ============================================================
import numpy as np, pandas as pd, pyfixest as pf
import matplotlib.pyplot as plt
from scipy.stats import norm
import warnings; warnings.filterwarnings('ignore')

GRID = 2                       # main 2; 1 for sensitivity
PANEL = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid{GRID}/panel_global.parquet'
panel = pd.read_parquet(PANEL)

# derived: cell season column (for grid×month FE)
panel['ym']    = panel['year_month']
panel['month'] = panel['year_month'].str[-2:]

print(f'panel: {panel.shape[0]:,} rows | {panel.region.nunique()} regions | '
      f"{panel.grid_id.nunique()} cells | accidents={panel.A_it.sum():,}")
print(panel.groupby('region').agg(obs=('A_it','size'), cells=('grid_id','nunique'),
      acc=('A_it','sum')).to_string())
EXTR = ['wind_only','wave_only','compound','wind_or_wave']
print('\nmean extreme days (days/cell-month):')
print(panel[[f'{c}_P95' for c in EXTR]+[f'{c}_P99' for c in EXTR]].mean().round(3).to_string())


In [ ]:
# ============================================================
# A2 — main PPML: A_it ~ wind_only + wave_only + compound
#   two-way FE: grid_id + region^year_month | offset log_E | clustered by grid_id
#   key test: is compound super-additive (β_comp > β_wind + β_wave)
# ============================================================
FE_MAIN = 'grid_id + region^year_month'

def stars(p): return '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '†' if p<.1 else ''

def superadd_test(res, tag):
    """Test H0: β_compound = β_wind_only + β_wave_only"""
    names = list(res.coef().index); b = res.coef().values; V = res._vcov
    c = np.zeros(len(names))
    c[names.index(f'compound_{tag}')]  = 1
    c[names.index(f'wind_only_{tag}')] = -1
    c[names.index(f'wave_only_{tag}')] = -1
    diff = float(c@b); se = float(np.sqrt(c@V@c)); z = diff/se
    p = 2*(1-norm.cdf(abs(z)))
    return diff, se, z, p

def run_main(panel, tag, fe=FE_MAIN):
    xs = [f'wind_only_{tag}', f'wave_only_{tag}', f'compound_{tag}']
    fml = f"A_it ~ {' + '.join(xs)} | {fe}"
    m = pf.fepois(fml, data=panel, offset='log_E', vcov={'CRV1':'grid_id'})
    print(f"\n{'='*66}\n  MAIN [{tag}]  {fml}\n{'='*66}")
    for x in xs:
        c,se,p = float(m.coef()[x]), float(m.se()[x]), float(m.pvalue()[x])
        print(f"    {x:<16} IRR/day={np.exp(c):.4f} "
              f"[{np.exp(c-1.96*se):.4f},{np.exp(c+1.96*se):.4f}] p={p:.4f}{stars(p)}")
    diff,se,z,p = superadd_test(m, tag)
    print(f"  ── super-additivity (compound − wind_only − wave_only) ──")
    print(f"    Δlog={diff:+.4f} (se={se:.4f})  ratio=exp(Δ)={np.exp(diff):.3f}  "
          f"z={z:+.2f} p={p:.4f}{stars(p)}  "
          f"→ {'SUPER-additive' if diff>0 and p<.1 else 'SUB-additive' if diff<0 and p<.1 else '≈ additive'}")
    return m

m95 = run_main(panel, 'P95')
m99 = run_main(panel, 'P99')


In [ ]:
# ============================================================
# A3 — main model per region (within-region FE: grid_id + year_month)
# ============================================================
def run_by_region(panel, tag='P95', min_acc=5):
    xs = [f'wind_only_{tag}', f'wave_only_{tag}', f'compound_{tag}']
    rows = []
    for r, g in panel.groupby('region'):
        if g.A_it.sum() < min_acc: 
            print(f'  {r:12s} SKIP (accidents={g.A_it.sum()})'); continue
        try:
            m = pf.fepois(f"A_it ~ {' + '.join(xs)} | grid_id + year_month",
                          data=g, offset='log_E', vcov={'CRV1':'grid_id'})
            d,_,_,pp = superadd_test(m, tag)
            print(f"  {r:12s} acc={int(g.A_it.sum()):>5} | " + " | ".join(
                f"{x.split('_')[0][:4]}:{np.exp(m.coef()[x]):.3f}{stars(m.pvalue()[x])}" for x in xs)
                + f" | superadd exp(Δ)={np.exp(d):.2f}{stars(pp)}")
            rows.append(dict(region=r, **{x:float(m.coef()[x]) for x in xs}))
        except Exception as e:
            print(f"  {r:12s} ERROR: {e}")
    return pd.DataFrame(rows)

print('=== main model by region (P95) ===')
reg_tab = run_by_region(panel, 'P95')


In [ ]:
# ============================================================
# A4 — spec ladder: S1 current | S2+grid×mo | S3+region×ym | S4 free logE | S5 no traffic
# ============================================================
def run_ladder(panel, tag='P95'):
    xs = f"wind_only_{tag} + wave_only_{tag} + compound_{tag}"
    SPECS = [
        ('S1 baseline',   f'A_it ~ {xs} | grid_id + year_month',                  dict(offset='log_E')),
        ('S2 +grid×mo',   f'A_it ~ {xs} | grid_id^month + year_month',            dict(offset='log_E')),
        ('S3 +region×ym', f'A_it ~ {xs} | grid_id^month + region^year_month',     dict(offset='log_E')),
        ('S4 free logE',  f'A_it ~ {xs} + log_E | grid_id^month + region^year_month', dict()),
        ('S5 no traffic', f'A_it ~ {xs} | grid_id^month + region^year_month',     dict()),
    ]
    print(f"\n{'='*66}\n  Spec Ladder [{tag}] — compound coefficient only\n{'='*66}")
    for lab, fml, kw in SPECS:
        try:
            m = pf.fepois(fml, data=panel, vcov={'CRV1':'grid_id'}, **kw)
            x = f'compound_{tag}'
            c,se,p = float(m.coef()[x]), float(m.se()[x]), float(m.pvalue()[x])
            extra = f"  θ_logE={float(m.coef()['log_E']):.3f}" if 'log_E' in m.coef().index else ''
            print(f"  {lab:<14} compound IRR={np.exp(c):.4f} "
                  f"[{np.exp(c-1.96*se):.4f},{np.exp(c+1.96*se):.4f}] p={p:.4f}{stars(p)}{extra}")
        except Exception as e:
            print(f"  {lab:<14} ERROR: {e}")

run_ladder(panel, 'P95')


In [ ]:
# ============================================================
# A5 — robustness: two-way clustered SE + leave-one-region-out
# ============================================================
FML_S3 = f'A_it ~ wind_only_P95 + wave_only_P95 + compound_P95 | grid_id^month + region^year_month'

print('=== S3 one-way vs two-way clustering (compound_P95) ===')
for lab, vc in [('one-way(grid)', {'CRV1':'grid_id'}),
                ('two-way',       {'CRV1':'grid_id+year_month'})]:
    m = pf.fepois(FML_S3, data=panel, offset='log_E', vcov=vc)
    x='compound_P95'; c,se,p=float(m.coef()[x]),float(m.se()[x]),float(m.pvalue()[x])
    print(f"  {lab:<16} IRR={np.exp(c):.4f} [{np.exp(c-1.96*se):.4f},{np.exp(c+1.96*se):.4f}] p={p:.4f}{stars(p)}")

print('\n=== leave-one-region-out (S3, two-way) ===')
base = pf.fepois(FML_S3, data=panel, offset='log_E', vcov={'CRV1':'grid_id+year_month'})
print(f"  full sample     compound IRR={np.exp(base.coef()['compound_P95']):.4f}")
for r in panel.region.unique():
    sub = panel[panel.region != r]
    try:
        m = pf.fepois(FML_S3, data=sub, offset='log_E', vcov={'CRV1':'grid_id+year_month'})
        print(f"  −{r:12s}   compound IRR={np.exp(m.coef()['compound_P95']):.4f}")
    except Exception as e:
        print(f"  −{r:12s}   ERROR: {e}")


In [ ]:
# ============================================================
# A6 — union summary (run separately, not with the three exclusive classes) + main-result forest plot
# ============================================================
for tag in ['P95','P99']:
    m = pf.fepois(f'A_it ~ wind_or_wave_{tag} | {FE_MAIN}',
                  data=panel, offset='log_E', vcov={'CRV1':'grid_id'})
    x=f'wind_or_wave_{tag}'; c,se,p=float(m.coef()[x]),float(m.se()[x]),float(m.pvalue()[x])
    print(f"  wind_or_wave_{tag}: IRR/day={np.exp(c):.4f} "
          f"[{np.exp(c-1.96*se):.4f},{np.exp(c+1.96*se):.4f}] p={p:.4f}{stars(p)}")

# forest plot: P95 three classes IRR/day
fig, ax = plt.subplots(figsize=(7,3.2))
xs = ['wind_only_P95','wave_only_P95','compound_P95']
for i,x in enumerate(xs):
    c,se = float(m95.coef()[x]), float(m95.se()[x])
    ax.errorbar(np.exp(c), i, xerr=[[np.exp(c)-np.exp(c-1.96*se)],[np.exp(c+1.96*se)-np.exp(c)]],
                fmt='o', capsize=4, color='#E15759' if 'compound' in x else '#4C78A8')
ax.axvline(1, ls='--', c='gray'); ax.set_yticks(range(3)); ax.set_yticklabels(xs)
ax.set(xlabel='IRR per extreme day', title='M2 main results (P95)'); ax.grid(alpha=.3, axis='x')
plt.tight_layout(); plt.show()


In [ ]:
# old ExtrDays_R95 ≈ new wave_only + compound (any extreme wave day)
panel['wave_ext_P95'] = panel['wave_only_P95'] + panel['compound_P95']

def repro_0722(panel, tag='P95', min_acc=3):
    print(f'=== replicate 0722 definition: A_it ~ wave_ext_{tag} | grid_id+year_month, active≥{min_acc} ===')
    for r, g in panel.groupby('region'):
        gg = g[g.groupby('grid_id').A_it.transform('sum') >= min_acc]
        if gg.A_it.sum() < 5:
            print(f'  {r:12s} skip (acc<5)'); continue
        m = pf.fepois(f'A_it ~ wave_ext_{tag} | grid_id + year_month',
                      data=gg, offset='log_E', vcov={'CRV1':'grid_id'})
        x=f'wave_ext_{tag}'; c,se,p=float(m.coef()[x]),float(m.se()[x]),float(m.pvalue()[x])
        print(f'  {r:12s} IRR/day={np.exp(c):.4f} '
              f'[{np.exp(c-1.96*se):.4f},{np.exp(c+1.96*se):.4f}] p={p:.4f}  '
              f'cells={gg.grid_id.nunique()} acc={int(gg.A_it.sum())}')

repro_0722(panel, 'P95')


In [ ]:
import os
CACHE = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid{GRID}'   # GRID defined in A1

def build_wave_ext_monthly(region, q=0.95):
    """regional P95 threshold per calendar month → count of any extreme wave day (replicates 0722 ExtrDays_R95 definition)"""
    w = pd.read_parquet(os.path.join(CACHE, f'{region}__dmax_wave.parquet')) \
          .rename(columns={'dmax': 'wave'})
    w['date']       = pd.to_datetime(w['date'])
    w['month']      = w.date.dt.month
    w['year_month'] = w.date.dt.strftime('%Y-%m')
    thr = w.groupby('month').wave.quantile(q)                 # region × calendar month threshold
    w['ext'] = (w.wave.values >= w.month.map(thr).values).astype('int8')
    agg = (w.groupby(['grid_id', 'year_month']).ext.sum()
             .rename('wave_ext_mth').reset_index())
    agg['region'] = region
    return agg

regions = list(panel.region.unique())
wem = pd.concat([build_wave_ext_monthly(r) for r in regions], ignore_index=True)
p2  = panel.merge(wem, on=['region', 'grid_id', 'year_month'], how='left')
p2['wave_ext_mth'] = p2['wave_ext_mth'].fillna(0)

print('=== isolation test: calendar-month threshold wave_ext_mth (otherwise as repro_0722) ===')
print(f'{"region":12s} {"cal. month (new test)":>20s}   vs  single annual (repro_0722)')
annual = {'SE_Asia':1.074,'E_Asia':1.076,'Med_Black':1.036,
          'NW_Atlantic':1.073,'NW_Europe':1.024,'Baltic':1.010}
for r, g in p2.groupby('region'):
    gg = g[g.groupby('grid_id').A_it.transform('sum') >= 3]
    if gg.A_it.sum() < 5:
        print(f'  {r:12s} skip'); continue
    m = pf.fepois('A_it ~ wave_ext_mth | grid_id + year_month',
                  data=gg, offset='log_E', vcov={'CRV1': 'grid_id'})
    x='wave_ext_mth'; c,se,p=float(m.coef()[x]),float(m.se()[x]),float(m.pvalue()[x])
    print(f'  {r:12s} IRR/day={np.exp(c):.4f} '
          f'[{np.exp(c-1.96*se):.4f},{np.exp(c+1.96*se):.4f}] p={p:.4f}   '
          f'(single annual={annual.get(r,float("nan")):.3f})')


In [ ]:
# ============================================================
# diagnostics: accident attrition at each stage + where they are lost (do regions miss many?)
# ============================================================
acc = pd.read_parquet(rf'{DATA_ROOT}/AIS_processing/accident_data/accident_cargo_clean_2002_2021.parquet')
acc = acc.dropna(subset=['lat','lon'])
acc = acc[acc.year.between(2011,2021)].copy()
N0 = len(acc)
acc['region'] = [assign_region(a,o) for a,o in zip(acc.lat, acc.lon)]

# ① region bounds
inreg = acc[acc.region.notna()].copy()
out   = acc[acc.region.isna()].copy()
print(f'① 2011-2021 valid accidents: {N0:,}')
print(f'② inside 6 regions: {len(inreg):,} ({100*len(inreg)/N0:.1f}%) | dropped outside: {len(out):,} ({100*len(out)/N0:.1f}%)')

# ② sea cell
clat,clon = to_cell(inreg.lat.values, inreg.lon.values)
inreg['grid_id'] = [gid(a,o) for a,o in zip(clat,clon)]
sea_set = set(sea_cells.grid_id)
in_sea = inreg[inreg.grid_id.isin(sea_set)].copy()
land   = inreg[~inreg.grid_id.isin(sea_set)].copy()
print(f'③ in sea cell: {len(in_sea):,} | dropped in non-sea cell (land/no SWH): {len(land):,}')

# ③ final panel (cell has traffic that month)
in_sea['year_month'] = pd.to_datetime(in_sea['casualty_date']).dt.strftime('%Y-%m')
pkeys = set(zip(panel.grid_id, panel.year_month))
in_panel = in_sea[[ (g,ym) in pkeys for g,ym in zip(in_sea.grid_id, in_sea.year_month)]]
notraf = len(in_sea) - len(in_panel)
print(f'④ in final panel: {len(in_panel):,}')
print(f'\n★ total attrition {N0-len(in_panel):,} ({100*(N0-len(in_panel))/N0:.1f}%) = '
      f'outside {len(out):,} + non-sea cell {len(land):,} + no traffic {notraf:,}')

# ④ where the out-of-region dropped accidents are (by casualty_location)
print('\n=== top 20 locations dropped outside regions ===')
print(out['casualty_location'].value_counts().head(20).to_string())


In [ ]:
# map: kept (green) vs dropped outside (red) + bounds (blue) — check for red clusters hugging the bounds
fig, ax = plt.subplots(figsize=(14,6))
ax.scatter(out.lon, out.lat, s=7, c='red',   alpha=.35, label=f'dropped outside {len(out)}')
ax.scatter(land.lon, land.lat, s=7, c='orange', alpha=.5, label=f'non-sea cell {len(land)}')
ax.scatter(in_panel.lon, in_panel.lat, s=5, c='green', alpha=.3, label=f'kept {len(in_panel)}')
draw_region_boxes(ax, color='blue', lw=1.2)
ax.legend(loc='lower left'); ax.grid(alpha=.3)
ax.set(title='Accidents kept/dropped + region bounds', xlabel='lon', ylabel='lat')
plt.tight_layout(); plt.show()
